# Python Design Pattern 실습

디자인 패턴은 크게 3가지 패턴으로 분류됩니다:
- **생성 (Creational)**: 객체를 어떻게 만들지
- **구조 (Structural)**: 객체들을 어떻게 연결하여 더 큰 구조를 만들지
- **행동 (Behavioral)**: 객체들이 어떻게 협력하고 행동을 바꾸는지

---
## 1. 생성 (Creational) 패턴

생성 패턴은 "객체를 어떻게 만들지"에 관한 설계법입니다.

### 1-1. Singleton 패턴

**목적**: 프로젝트 내에 단 하나만 존재해야 하는 경우 사용

**사용 시기**:
- 프로그램 전체에서 하나로 공유해야 할 객체
- 전역에서 같은 객체를 쉽게 재사용

**머신러닝 엔지니어 관점:**
ML 프로젝트에서 모델 레지스트리, 설정 관리자, 로깅 시스템 등에 유용합니다. 예를 들어, 전역 모델 레지스트리에서 모델을 등록하고 검색할 때 사용.

**장점**: 전역 공유가 쉬움

**단점**: 너무 많이 사용 시 전역 변수처럼 되어 테스트/병렬 처리 시 문제 발생 가능

In [1]:
# Singleton 패턴 더 자세한 예제: ML 모델 레지스트리
# 머신러닝에서 Singleton 패턴 적용 예시

class ModelRegistry:
    """ML 모델을 전역에서 관리하는 싱글톤 레지스트리"""
    _instance = None
    
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._models = {}  # 모델 저장소
            cls._instance._metadata = {}  # 메타데이터 저장소
        return cls._instance
    
    def register_model(self, name, model, metadata=None):
        """모델 등록"""
        self._models[name] = model
        self._metadata[name] = metadata or {}
        print(f"✓ 모델 '{name}' 등록됨")
    
    def get_model(self, name):
        """모델 조회"""
        if name not in self._models:
            raise ValueError(f"모델 '{name}'이 등록되지 않았습니다")
        return self._models[name]
    
    def list_models(self):
        """등록된 모델 목록"""
        return list(self._models.keys())
    
    def get_model_info(self, name):
        """모델 정보 조회"""
        if name not in self._models:
            return None
        return {
            'name': name,
            'metadata': self._metadata[name],
            'type': type(self._models[name]).__name__
        }
    
    def update_metadata(self, name, metadata):
        """메타데이터 업데이트"""
        if name in self._metadata:
            self._metadata[name].update(metadata)
            print(f"✓ 모델 '{name}' 메타데이터 업데이트됨")

# 테스트: 싱글톤 레지스트리 사용
print("=== ML 모델 레지스트리 데모 ===")

# 서로 다른 인스턴스지만 실제로는 같은 객체
registry1 = ModelRegistry()
registry2 = ModelRegistry()

print(f"registry1 is registry2: {registry1 is registry2}")  # True

# 모델 등록 (어떤 인스턴스든 동일)
registry1.register_model(
    "sentiment_classifier", 
    "RandomForest 모델 객체", 
    {"accuracy": 0.85, "version": "1.0", "author": "ml_team"}
)

registry2.register_model(
    "image_classifier", 
    "CNN 모델 객체", 
    {"accuracy": 0.92, "version": "2.1", "framework": "PyTorch"}
)

# 모델 조회 (다른 인스턴스에서도 동일한 데이터)
print(f"\n등록된 모델들: {registry2.list_models()}")

model_info = registry1.get_model_info("sentiment_classifier")
print(f"감정 분류기 정보: {model_info}")

# 메타데이터 업데이트
registry2.update_metadata("sentiment_classifier", {"last_updated": "2024-01-15"})

updated_info = registry1.get_model_info("sentiment_classifier")
print(f"업데이트된 정보: {updated_info}")

print("\n🎯 Singleton 패턴으로 전역 모델 레지스트리 구현!")
print("   어느 곳에서나 동일한 모델 관리 가능")

=== ML 모델 레지스트리 데모 ===
registry1 is registry2: True
✓ 모델 'sentiment_classifier' 등록됨
✓ 모델 'image_classifier' 등록됨

등록된 모델들: ['sentiment_classifier', 'image_classifier']
감정 분류기 정보: {'name': 'sentiment_classifier', 'metadata': {'accuracy': 0.85, 'version': '1.0', 'author': 'ml_team'}, 'type': 'str'}
✓ 모델 'sentiment_classifier' 메타데이터 업데이트됨
업데이트된 정보: {'name': 'sentiment_classifier', 'metadata': {'accuracy': 0.85, 'version': '1.0', 'author': 'ml_team', 'last_updated': '2024-01-15'}, 'type': 'str'}

🎯 Singleton 패턴으로 전역 모델 레지스트리 구현!
   어느 곳에서나 동일한 모델 관리 가능


### 1-2. Factory Method 패턴

**목적**: 어떤 객체를 생성할 때, 어떤 클래스의 인스턴스를 생성할지 결정하는 책임을 따로 분리

**사용 시기**:
- 여러 종류의 객체 중 어떤 것을 생성할지 Runtime에 결정

**장점**: 객체 생성 코드를 분리하여 코드 변경이 쉬움

**단점**: Factory Class/Method가 증가하면 관리가 복잡해짐

In [2]:
# Factory Method 패턴 예제: ML 모델 생성 팩토리
# 머신러닝에서 Factory Method 패턴 적용 예시

class MLModel:
    """기본 ML 모델 인터페이스"""
    def __init__(self):
        self.name = "기본 ML 모델"
        self.framework = "Unknown"
    
    def train(self, X, y):
        """모델 학습"""
        pass
    
    def predict(self, X):
        """예측"""
        pass

class LinearRegressionModel(MLModel):
    """선형 회귀 모델"""
    def __init__(self):
        self.name = "선형 회귀 모델"
        self.framework = "scikit-learn"
        self.model = None  # 실제로는 sklearn.linear_model.LinearRegression()
    
    def train(self, X, y):
        print(f"✓ {self.name} 학습 중...")
        # 실제로는: self.model.fit(X, y)
        return f"{self.name} 학습 완료"
    
    def predict(self, X):
        print(f"✓ {self.name} 예측 중...")
        # 실제로는: return self.model.predict(X)
        return "예측 결과"

class RandomForestModel(MLModel):
    """랜덤 포레스트 모델"""
    def __init__(self):
        self.name = "랜덤 포레스트 모델"
        self.framework = "scikit-learn"
        self.model = None  # 실제로는 sklearn.ensemble.RandomForestClassifier()
    
    def train(self, X, y):
        print(f"✓ {self.name} 학습 중...")
        # 실제로는: self.model.fit(X, y)
        return f"{self.name} 학습 완료"
    
    def predict(self, X):
        print(f"✓ {self.name} 예측 중...")
        # 실제로는: return self.model.predict(X)
        return "예측 결과"

class NeuralNetworkModel(MLModel):
    """신경망 모델"""
    def __init__(self):
        self.name = "신경망 모델"
        self.framework = "PyTorch"
        self.model = None  # 실제로는 torch.nn.Module
    
    def train(self, X, y):
        print(f"✓ {self.name} 학습 중...")
        # 실제로는: optimizer, loss function 등 설정 후 학습
        return f"{self.name} 학습 완료"
    
    def predict(self, X):
        print(f"✓ {self.name} 예측 중...")
        # 실제로는: return self.model(X)
        return "예측 결과"

def create_ml_model(model_type, **kwargs):
    """
    Factory Method: 모델 타입에 따라 적절한 모델 생성
    
    Args:
        model_type (str): 모델 타입 ('linear', 'rf', 'nn')
        **kwargs: 모델 초기화 파라미터
    """
    if model_type == "linear":
        return LinearRegressionModel()
    elif model_type == "rf":
        return RandomForestModel()
    elif model_type == "nn":
        return NeuralNetworkModel()
    else:
        raise ValueError(f"지원하지 않는 모델 타입: {model_type}")

# 테스트: ML 모델 팩토리 사용
print("=== ML 모델 생성 팩토리 데모 ===")

# 다양한 모델 생성
linear_model = create_ml_model("linear")
rf_model = create_ml_model("rf")
nn_model = create_ml_model("nn")

print(f"생성된 모델들:")
print(f"1. {linear_model.name} ({linear_model.framework})")
print(f"2. {rf_model.name} ({rf_model.framework})")
print(f"3. {nn_model.name} ({nn_model.framework})")

# 모델 학습 및 예측 데모
print(f"\n{linear_model.train('X_train', 'y_train')}")
print(f"{linear_model.predict('X_test')}")

print(f"\n{rf_model.train('X_train', 'y_train')}")
print(f"{rf_model.predict('X_test')}")

print("\n🎯 Factory Method 패턴으로 ML 모델 생성 로직 분리!")
print("   새로운 모델 추가 시 팩토리 함수만 수정하면 됨")

=== ML 모델 생성 팩토리 데모 ===
생성된 모델들:
1. 선형 회귀 모델 (scikit-learn)
2. 랜덤 포레스트 모델 (scikit-learn)
3. 신경망 모델 (PyTorch)
✓ 선형 회귀 모델 학습 중...

선형 회귀 모델 학습 완료
✓ 선형 회귀 모델 예측 중...
예측 결과
✓ 랜덤 포레스트 모델 학습 중...

랜덤 포레스트 모델 학습 완료
✓ 랜덤 포레스트 모델 예측 중...
예측 결과

🎯 Factory Method 패턴으로 ML 모델 생성 로직 분리!
   새로운 모델 추가 시 팩토리 함수만 수정하면 됨


### 1-3. Abstract Factory 패턴

**목적**: 서로 관련된 여러 객체를 하나의 묶음으로 만들어줌

**사용 시기**:
- 서로 연관되는 여러 객체를 함께 만들어야 할 때 (예: Windows용 UI, Mac용 UI)

**장점**: 일관된 기능을 보장

**단점**: 기능이 추가되면 Factory Interface 수정이 필요

In [3]:
# Abstract Factory 패턴 예제: ML 파이프라인 컴포넌트 팩토리
# 머신러닝에서 Abstract Factory 패턴 적용 예시

# 제품 인터페이스들
class DataPreprocessor:
    """데이터 전처리기 인터페이스"""
    def __init__(self):
        self.name = "기본 전처리기"
    
    def preprocess(self, data):
        pass

class MLModel:
    """ML 모델 인터페이스"""
    def __init__(self):
        self.name = "기본 모델"
    
    def train(self, X, y):
        pass
    
    def predict(self, X):
        pass

class Evaluator:
    """모델 평가기 인터페이스"""
    def __init__(self):
        self.name = "기본 평가기"
    
    def evaluate(self, y_true, y_pred):
        pass

# Scikit-learn 스타일 컴포넌트들
class SklearnPreprocessor(DataPreprocessor):
    def __init__(self):
        self.name = "Scikit-learn 전처리기"
    
    def preprocess(self, data):
        print("✓ Scikit-learn으로 데이터 전처리 중...")
        return "전처리된 데이터"

class SklearnModel(MLModel):
    def __init__(self):
        self.name = "Scikit-learn 모델"
    
    def train(self, X, y):
        print("✓ Scikit-learn 모델 학습 중...")
        return "학습된 모델"
    
    def predict(self, X):
        print("✓ Scikit-learn 모델 예측 중...")
        return "예측 결과"

class SklearnEvaluator(Evaluator):
    def __init__(self):
        self.name = "Scikit-learn 평가기"
    
    def evaluate(self, y_true, y_pred):
        print("✓ Scikit-learn 메트릭 계산 중...")
        return {"accuracy": 0.85, "precision": 0.82}

# PyTorch 스타일 컴포넌트들
class TorchPreprocessor(DataPreprocessor):
    def __init__(self):
        self.name = "PyTorch 전처리기"
    
    def preprocess(self, data):
        print("✓ PyTorch로 데이터 전처리 중...")
        return "전처리된 텐서 데이터"

class TorchModel(MLModel):
    def __init__(self):
        self.name = "PyTorch 모델"
    
    def train(self, X, y):
        print("✓ PyTorch 모델 학습 중...")
        return "학습된 신경망"
    
    def predict(self, X):
        print("✓ PyTorch 모델 예측 중...")
        return "예측 텐서"

class TorchEvaluator(Evaluator):
    def __init__(self):
        self.name = "PyTorch 평가기"
    
    def evaluate(self, y_true, y_pred):
        print("✓ PyTorch 메트릭 계산 중...")
        return {"accuracy": 0.88, "f1_score": 0.86}

# 추상 팩토리 인터페이스
class MLPipelineFactory:
    """ML 파이프라인 컴포넌트 팩토리 인터페이스"""
    def create_preprocessor(self):
        pass
    
    def create_model(self):
        pass
    
    def create_evaluator(self):
        pass

# 구체 팩토리들
class SklearnPipelineFactory(MLPipelineFactory):
    """Scikit-learn 기반 파이프라인 팩토리"""
    def create_preprocessor(self):
        return SklearnPreprocessor()
    
    def create_model(self):
        return SklearnModel()
    
    def create_evaluator(self):
        return SklearnEvaluator()

class TorchPipelineFactory(MLPipelineFactory):
    """PyTorch 기반 파이프라인 팩토리"""
    def create_preprocessor(self):
        return TorchPreprocessor()
    
    def create_model(self):
        return TorchModel()
    
    def create_evaluator(self):
        return TorchEvaluator()

# 테스트: ML 파이프라인 팩토리 사용
print("=== Scikit-learn 파이프라인 ===")
sklearn_factory = SklearnPipelineFactory()

preprocessor = sklearn_factory.create_preprocessor()
model = sklearn_factory.create_model()
evaluator = sklearn_factory.create_evaluator()

print(f"컴포넌트들: {preprocessor.name}, {model.name}, {evaluator.name}")

# 파이프라인 실행
processed_data = preprocessor.preprocess("raw_data")
trained_model = model.train(processed_data, "labels")
predictions = model.predict("test_data")
metrics = evaluator.evaluate("true_labels", predictions)

print(f"평가 결과: {metrics}")

print("\n=== PyTorch 파이프라인 ===")
torch_factory = TorchPipelineFactory()

preprocessor = torch_factory.create_preprocessor()
model = torch_factory.create_model()
evaluator = torch_factory.create_evaluator()

print(f"컴포넌트들: {preprocessor.name}, {model.name}, {evaluator.name}")

# 파이프라인 실행
processed_data = preprocessor.preprocess("raw_data")
trained_model = model.train(processed_data, "labels")
predictions = model.predict("test_data")
metrics = evaluator.evaluate("true_labels", predictions)

print(f"평가 결과: {metrics}")

print("\n🎯 Abstract Factory 패턴으로 ML 프레임워크별 컴포넌트 묶음 생성!")
print("   프레임워크 변경 시 팩토리만 교체하면 전체 파이프라인 일관성 유지")

=== Scikit-learn 파이프라인 ===
컴포넌트들: Scikit-learn 전처리기, Scikit-learn 모델, Scikit-learn 평가기
✓ Scikit-learn으로 데이터 전처리 중...
✓ Scikit-learn 모델 학습 중...
✓ Scikit-learn 모델 예측 중...
✓ Scikit-learn 메트릭 계산 중...
평가 결과: {'accuracy': 0.85, 'precision': 0.82}

=== PyTorch 파이프라인 ===
컴포넌트들: PyTorch 전처리기, PyTorch 모델, PyTorch 평가기
✓ PyTorch로 데이터 전처리 중...
✓ PyTorch 모델 학습 중...
✓ PyTorch 모델 예측 중...
✓ PyTorch 메트릭 계산 중...
평가 결과: {'accuracy': 0.88, 'f1_score': 0.86}

🎯 Abstract Factory 패턴으로 ML 프레임워크별 컴포넌트 묶음 생성!
   프레임워크 변경 시 팩토리만 교체하면 전체 파이프라인 일관성 유지


### 1-4. Builder 패턴

**목적**: 복잡한 객체를 단계별로 만들 때, 만들기 과정을 분리하여 유연하게 조립

**사용 시기**:
- 객체를 만드는 과정이 복잡하고 여러 옵션이 존재할 때

**장점**: 여러 종류의 생성 과정을 같은 빌더로 재사용 가능

**단점**: 단순한 객체 생성엔 오버엔지니어링일 수 있음

In [4]:
# Builder 패턴 예제: ML 모델 학습 파이프라인 빌더
# 머신러닝에서 Builder 패턴 적용 예시

class MLPipeline:
    """ML 학습 파이프라인 클래스"""
    def __init__(self):
        self.steps = []
        self.name = "ML 학습 파이프라인"
    
    def add_step(self, step_name, step_func):
        """파이프라인에 단계 추가"""
        self.steps.append((step_name, step_func))
        return self
    
    def execute(self, data=None):
        """파이프라인 실행"""
        result = data
        print(f"🚀 {self.name} 실행 시작")
        
        for step_name, step_func in self.steps:
            print(f"  → {step_name} 실행 중...")
            result = step_func(result)
        
        print(f"✅ {self.name} 실행 완료")
        return result

class MLPipelineBuilder:
    """ML 파이프라인 빌더"""
    def __init__(self):
        self.pipeline = MLPipeline()
    
    def set_name(self, name):
        """파이프라인 이름 설정"""
        self.pipeline.name = name
        return self
    
    def add_data_loading(self, source="csv"):
        """데이터 로딩 단계 추가"""
        def load_data(prev_result=None):
            print(f"    데이터 로딩: {source} 파일에서 데이터 읽기")
            return {"data": f"loaded_{source}_data", "metadata": {"source": source}}
        self.pipeline.add_step("데이터 로딩", load_data)
        return self
    
    def add_preprocessing(self, method="standard"):
        """데이터 전처리 단계 추가"""
        def preprocess(data):
            print(f"    데이터 전처리: {method} 방법 적용")
            if data:
                data["preprocessed"] = True
                data["preprocessing_method"] = method
            return data
        self.pipeline.add_step("데이터 전처리", preprocess)
        return self
    
    def add_feature_engineering(self, features=None):
        """특징 공학 단계 추가"""
        features = features or ["feature1", "feature2"]
        def engineer_features(data):
            print(f"    특징 공학: {features} 생성")
            if data:
                data["features"] = features
                data["feature_count"] = len(features)
            return data
        self.pipeline.add_step("특징 공학", engineer_features)
        return self
    
    def add_model_selection(self, model_type="rf"):
        """모델 선택 단계 추가"""
        def select_model(data):
            print(f"    모델 선택: {model_type} 모델 선택")
            if data:
                data["model_type"] = model_type
                data["model_config"] = {"n_estimators": 100} if model_type == "rf" else {"hidden_layers": [64, 32]}
            return data
        self.pipeline.add_step("모델 선택", select_model)
        return self
    
    def add_training(self, epochs=10):
        """모델 학습 단계 추가"""
        def train_model(data):
            print(f"    모델 학습: {epochs} 에폭으로 학습")
            if data:
                data["trained_model"] = f"trained_{data.get('model_type', 'unknown')}_model"
                data["training_epochs"] = epochs
                data["training_complete"] = True
            return data
        self.pipeline.add_step("모델 학습", train_model)
        return self
    
    def add_evaluation(self, metrics=None):
        """모델 평가 단계 추가"""
        metrics = metrics or ["accuracy", "precision"]
        def evaluate_model(data):
            print(f"    모델 평가: {metrics} 메트릭 계산")
            if data:
                data["evaluation_metrics"] = {metric: 0.85 + i*0.02 for i, metric in enumerate(metrics)}
                data["evaluation_complete"] = True
            return data
        self.pipeline.add_step("모델 평가", evaluate_model)
        return self
    
    def add_model_saving(self, path="./models"):
        """모델 저장 단계 추가"""
        def save_model(data):
            print(f"    모델 저장: {path}에 모델 저장")
            if data:
                data["model_path"] = f"{path}/{data.get('trained_model', 'model')}.pkl"
                data["saved"] = True
            return data
        self.pipeline.add_step("모델 저장", save_model)
        return self
    
    def build(self):
        """파이프라인 빌드"""
        return self.pipeline

# 테스트: ML 파이프라인 빌더 사용
print("=== 기본 ML 파이프라인 ===")
basic_pipeline = (MLPipelineBuilder()
    .set_name("기본 ML 파이프라인")
    .add_data_loading("csv")
    .add_preprocessing("standard")
    .add_model_selection("rf")
    .add_training(10)
    .add_evaluation()
    .build())

result1 = basic_pipeline.execute()
print(f"최종 결과: {result1}")

print("\n=== 고급 ML 파이프라인 ===")
advanced_pipeline = (MLPipelineBuilder()
    .set_name("고급 ML 파이프라인")
    .add_data_loading("database")
    .add_preprocessing("robust")
    .add_feature_engineering(["feature1", "feature2", "feature3", "feature4"])
    .add_model_selection("nn")
    .add_training(50)
    .add_evaluation(["accuracy", "f1_score", "precision", "recall"])
    .add_model_saving("./production_models")
    .build())

result2 = advanced_pipeline.execute()
print(f"최종 결과: {result2}")

print("\n🎯 Builder 패턴으로 유연한 ML 파이프라인 구성!")
print("   단계별로 필요한 컴포넌트만 선택하여 파이프라인 구축")

=== 기본 ML 파이프라인 ===
🚀 기본 ML 파이프라인 실행 시작
  → 데이터 로딩 실행 중...
    데이터 로딩: csv 파일에서 데이터 읽기
  → 데이터 전처리 실행 중...
    데이터 전처리: standard 방법 적용
  → 모델 선택 실행 중...
    모델 선택: rf 모델 선택
  → 모델 학습 실행 중...
    모델 학습: 10 에폭으로 학습
  → 모델 평가 실행 중...
    모델 평가: ['accuracy', 'precision'] 메트릭 계산
✅ 기본 ML 파이프라인 실행 완료
최종 결과: {'data': 'loaded_csv_data', 'metadata': {'source': 'csv'}, 'preprocessed': True, 'preprocessing_method': 'standard', 'model_type': 'rf', 'model_config': {'n_estimators': 100}, 'trained_model': 'trained_rf_model', 'training_epochs': 10, 'training_complete': True, 'evaluation_metrics': {'accuracy': 0.85, 'precision': 0.87}, 'evaluation_complete': True}

=== 고급 ML 파이프라인 ===
🚀 고급 ML 파이프라인 실행 시작
  → 데이터 로딩 실행 중...
    데이터 로딩: database 파일에서 데이터 읽기
  → 데이터 전처리 실행 중...
    데이터 전처리: robust 방법 적용
  → 특징 공학 실행 중...
    특징 공학: ['feature1', 'feature2', 'feature3', 'feature4'] 생성
  → 모델 선택 실행 중...
    모델 선택: nn 모델 선택
  → 모델 학습 실행 중...
    모델 학습: 50 에폭으로 학습
  → 모델 평가 실행 중...
    모델 평가: ['accuracy', 'f1_s

### 1-5. Prototype 패턴

**목적**: 이미 생성된 객체를 복사하여 새로운 객체를 만들 때 사용

**사용 시기**:
- 객체 생성 비용이 크거나 초기 설정이 복잡할 때
- 기존 객체를 복사하여 일부만 변경하고 싶을 때

**장점**: 복사로 빠른 객체 생성

**단점**: 참조를 공유할 위험이 존재

In [5]:
# Prototype 패턴 예제: ML 모델 설정 프로토타입
# 머신러닝에서 Prototype 패턴 적용 예시

import copy

class ModelConfig:
    """ML 모델 설정 클래스"""
    def __init__(self, model_type, hyperparameters=None, preprocessing=None, features=None):
        self.model_type = model_type
        self.hyperparameters = hyperparameters or {}
        self.preprocessing = preprocessing or {}
        self.features = features or []
        self.experiment_name = f"{model_type}_config"
    
    def __str__(self):
        return f"ModelConfig({self.model_type}, HP: {self.hyperparameters}, Features: {len(self.features)})"
    
    def clone(self, deep=True):
        """설정 복사 (프로토타입 패턴)"""
        if deep:
            return copy.deepcopy(self)
        else:
            return copy.copy(self)
    
    def update_hyperparameters(self, **kwargs):
        """하이퍼파라미터 업데이트"""
        self.hyperparameters.update(kwargs)
        return self
    
    def add_feature(self, feature):
        """특징 추가"""
        self.features.append(feature)
        return self
    
    def set_experiment_name(self, name):
        """실험 이름 설정"""
        self.experiment_name = name
        return self

# 기본 설정 프로토타입들 생성
print("=== ML 모델 설정 프로토타입 생성 ===")

# Random Forest 기본 설정
rf_base = ModelConfig(
    "RandomForest",
    hyperparameters={"n_estimators": 100, "max_depth": 10, "random_state": 42},
    preprocessing={"scaler": "StandardScaler", "imputer": "SimpleImputer"},
    features=["feature1", "feature2", "feature3"]
)

# Neural Network 기본 설정
nn_base = ModelConfig(
    "NeuralNetwork",
    hyperparameters={"hidden_layers": [64, 32], "learning_rate": 0.001, "epochs": 100},
    preprocessing={"scaler": "MinMaxScaler", "encoder": "OneHotEncoder"},
    features=["feature1", "feature2", "feature3", "feature4"]
)

print(f"기본 RF 설정: {rf_base}")
print(f"기본 NN 설정: {nn_base}")

# 얕은 복사로 실험 설정 생성 (참조 공유 주의)
print("\n=== 얕은 복사로 실험 설정 생성 ===")
rf_exp1 = rf_base.clone(deep=False)
rf_exp1.set_experiment_name("rf_tuned_max_depth")
rf_exp1.update_hyperparameters(max_depth=20)

rf_exp2 = rf_base.clone(deep=False)
rf_exp2.set_experiment_name("rf_tuned_estimators")
rf_exp2.update_hyperparameters(n_estimators=200)

print(f"실험 1: {rf_exp1}")
print(f"실험 2: {rf_exp2}")

# 특징 추가 시 원본에 영향 (얕은 복사의 문제점)
print("\n=== 얕은 복사의 문제점 ===")
print(f"원본 특징 수: {len(rf_base.features)}")
rf_exp1.add_feature("feature4")  # 실험에서 특징 추가
print(f"실험 1 특징 수: {len(rf_exp1.features)}")
print(f"원본 특징 수: {len(rf_base.features)}")  # 원본도 변경됨!

# 깊은 복사로 안전한 실험 설정 생성
print("\n=== 깊은 복사로 안전한 실험 설정 생성 ===")
nn_exp1 = nn_base.clone(deep=True)
nn_exp1.set_experiment_name("nn_deeper_network")
nn_exp1.update_hyperparameters(hidden_layers=[128, 64, 32])

nn_exp2 = nn_base.clone(deep=True)
nn_exp2.set_experiment_name("nn_faster_learning")
nn_exp2.update_hyperparameters(learning_rate=0.01, epochs=50)

print(f"실험 1: {nn_exp1}")
print(f"실험 2: {nn_exp2}")

# 특징 추가 시 원본 영향 없음
print("\n=== 깊은 복사의 장점 ===")
print(f"원본 특징 수: {len(nn_base.features)}")
nn_exp1.add_feature("feature5")  # 실험에서 특징 추가
print(f"실험 1 특징 수: {len(nn_exp1.features)}")
print(f"원본 특징 수: {len(nn_base.features)}")  # 원본 영향 없음

# 여러 실험 설정을 한 번에 생성하는 헬퍼 함수
def create_experiments_from_prototype(prototype, experiments_config):
    """프로토타입으로부터 여러 실험 설정 생성"""
    experiments = []
    for exp_name, changes in experiments_config.items():
        exp = prototype.clone(deep=True)
        exp.set_experiment_name(exp_name)
        if "hyperparameters" in changes:
            exp.update_hyperparameters(**changes["hyperparameters"])
        if "features" in changes:
            for feature in changes["features"]:
                exp.add_feature(feature)
        experiments.append(exp)
    return experiments

# RF 실험 설정들 생성
rf_experiments_config = {
    "rf_shallow": {"hyperparameters": {"max_depth": 5}},
    "rf_deep": {"hyperparameters": {"max_depth": 20}},
    "rf_many_trees": {"hyperparameters": {"n_estimators": 500}},
    "rf_with_extra_features": {"hyperparameters": {"n_estimators": 150}, "features": ["feature5", "feature6"]}
}

rf_experiments = create_experiments_from_prototype(rf_base, rf_experiments_config)

print("\n=== 프로토타입으로 생성된 RF 실험들 ===")
for exp in rf_experiments:
    print(f"  {exp}")

print("\n🎯 Prototype 패턴으로 ML 모델 설정 템플릿 재사용!")
print("   깊은 복사로 안전하게 설정 변형 및 실험 생성")

=== ML 모델 설정 프로토타입 생성 ===
기본 RF 설정: ModelConfig(RandomForest, HP: {'n_estimators': 100, 'max_depth': 10, 'random_state': 42}, Features: 3)
기본 NN 설정: ModelConfig(NeuralNetwork, HP: {'hidden_layers': [64, 32], 'learning_rate': 0.001, 'epochs': 100}, Features: 4)

=== 얕은 복사로 실험 설정 생성 ===
실험 1: ModelConfig(RandomForest, HP: {'n_estimators': 200, 'max_depth': 20, 'random_state': 42}, Features: 3)
실험 2: ModelConfig(RandomForest, HP: {'n_estimators': 200, 'max_depth': 20, 'random_state': 42}, Features: 3)

=== 얕은 복사의 문제점 ===
원본 특징 수: 3
실험 1 특징 수: 4
원본 특징 수: 4

=== 깊은 복사로 안전한 실험 설정 생성 ===
실험 1: ModelConfig(NeuralNetwork, HP: {'hidden_layers': [128, 64, 32], 'learning_rate': 0.001, 'epochs': 100}, Features: 4)
실험 2: ModelConfig(NeuralNetwork, HP: {'hidden_layers': [64, 32], 'learning_rate': 0.01, 'epochs': 50}, Features: 4)

=== 깊은 복사의 장점 ===
원본 특징 수: 4
실험 1 특징 수: 5
원본 특징 수: 4

=== 프로토타입으로 생성된 RF 실험들 ===
  ModelConfig(RandomForest, HP: {'n_estimators': 200, 'max_depth': 5, 'random_state': 42}, 

---
## 2. 구조 (Structural) 패턴

구조 패턴은 "객체들을 어떻게 잘 연결하여 더 큰 구조를 만들지"에 관련된 방법입니다.

### 2-1. Adapter 패턴

**목적**: 인터페이스가 다른 두 객체를 연결해주는 변환기 역할

**사용 시기**:
- 이미 존재하는 Class의 인터페이스와 내 코드가 기대하는 인터페이스가 다를 때
- 기존 코드를 고치지 않고 호환시키고 싶을 때

**머신러닝 엔지니어 관점:**
ML 프로젝트에서 서로 다른 라이브러리나 API의 인터페이스를 통일할 때 유용합니다. 예를 들어, scikit-learn과 TensorFlow의 모델 인터페이스를 동일하게 만들어주는 어댑터.

**장점**: 기존 코드 수정 없이 호환 가능

**단점**: Adapter가 많아지면 코드가 복잡해질 수 있음

In [6]:
# Adapter 패턴 더 자세한 예제: ML 모델 인터페이스 통일
# 상황: 서로 다른 ML 라이브러리의 모델 인터페이스가 다름

# 1. 표준화된 ML 모델 인터페이스 (내가 원하는 인터페이스)
class MLModel:
    def fit(self, X, y):
        """모델 학습"""
        pass
    
    def predict(self, X):
        """예측"""
        pass
    
    def score(self, X, y):
        """평가 점수 반환"""
        pass

# 2. 기존 scikit-learn 스타일 모델 (호환되지 않는 인터페이스)
class SklearnModel:
    def train(self, X_train, y_train):
        """scikit-learn 스타일 학습"""
        print("✓ Sklearn 모델 학습 중...")
        return "trained_sklearn_model"
    
    def predict_proba(self, X_test):
        """확률 예측"""
        print("✓ Sklearn 모델 확률 예측 중...")
        return [0.8, 0.2, 0.9]  # 예시 확률
    
    def evaluate(self, X_val, y_val):
        """평가"""
        print("✓ Sklearn 모델 평가 중...")
        return 0.85  # accuracy

# 3. 어댑터 클래스: SklearnModel을 MLModel 인터페이스로 변환
class SklearnAdapter(MLModel):
    def __init__(self, sklearn_model):
        self.sklearn_model = sklearn_model
    
    def fit(self, X, y):
        # 인터페이스 변환: fit -> train
        return self.sklearn_model.train(X, y)
    
    def predict(self, X):
        # 인터페이스 변환: predict_proba -> predict (클래스 예측으로 변환)
        proba = self.sklearn_model.predict_proba(X)
        return [1 if p > 0.5 else 0 for p in proba]  # 임계값 0.5로 이진 분류
    
    def score(self, X, y):
        # 인터페이스 변환: evaluate -> score
        return self.sklearn_model.evaluate(X, y)

# 테스트
print("=== 기존 Sklearn 모델 직접 사용 ===")
sklearn_model = SklearnModel()
sklearn_model.train([[1, 2], [3, 4]], [0, 1])  # 인터페이스가 다름
proba = sklearn_model.predict_proba([[5, 6]])
print(f"확률 예측: {proba}")

print("\n=== 어댑터를 통해 표준 인터페이스로 사용 ===")
adapter = SklearnAdapter(sklearn_model)
adapter.fit([[1, 2], [3, 4]], [0, 1])  # 표준 인터페이스
predictions = adapter.predict([[5, 6]])
print(f"클래스 예측: {predictions}")
accuracy = adapter.score([[1, 2], [3, 4]], [0, 1])
print(f"정확도: {accuracy}")

# 이제 MLModel 인터페이스를 구현한 어떤 모델이든 동일하게 사용할 수 있음!
print("\n🎯 Adapter 패턴으로 ML 라이브러리 인터페이스 통일!")
print("   서로 다른 라이브러리의 모델을 동일한 방식으로 사용 가능")

=== 기존 Sklearn 모델 직접 사용 ===
✓ Sklearn 모델 학습 중...
✓ Sklearn 모델 확률 예측 중...
확률 예측: [0.8, 0.2, 0.9]

=== 어댑터를 통해 표준 인터페이스로 사용 ===
✓ Sklearn 모델 학습 중...
✓ Sklearn 모델 확률 예측 중...
클래스 예측: [1, 0, 1]
✓ Sklearn 모델 평가 중...
정확도: 0.85

🎯 Adapter 패턴으로 ML 라이브러리 인터페이스 통일!
   서로 다른 라이브러리의 모델을 동일한 방식으로 사용 가능


### 2-2. Bridge 패턴
Bridge 패턴은 **"추상(인터페이스)과 구현을 분리하여 독립적으로 확장할 수 있게 하는 패턴"**입니다.

**비유로 이해하기:**
- **추상(Abstraction)**: ML 모델 (분류기, 회귀 모델 등) - "어떤 모델을 사용할까?"
- **구현(Implementation)**: 데이터 저장 방식 (메모리, 파일, 데이터베이스) - "데이터를 어떻게 저장할까?"

모델은 저장 방식을 모르고, 저장 방식은 모델을 모릅니다. 서로 독립적으로 변경 가능합니다.

**머신러닝 엔지니어 관점:**
ML 시스템에서 모델 로직과 데이터 저장/로딩 로직을 분리할 때 유용합니다. 모델을 바꿔도 저장 방식은 그대로, 저장 방식을 바꿔도 모델은 그대로 유지됩니다.

**코드로 이해하기:**
- `MLModel`: 추상 클래스 (모델의 개념)
- `DataStorage`: 구현 인터페이스 (저장 방식)
- `Classifier`: 구체 추상 (특정 모델)
- `MemoryStorage`: 구체 구현 (특정 저장 방식)

**장점:**
- 모델 종류를 늘려도 저장 방식은 그대로
- 저장 방식을 바꿔도 모델 코드는 수정 불필요
- 조합의 자유도 증가 (모델 × 저장방식)

**단점:**
- 코드가 약간 복잡해짐 (단순한 경우엔 오버엔지니어링)

In [7]:
# Bridge 패턴 더 자세한 예제: ML 모델과 데이터 저장소 분리
# 모델 로직과 데이터 저장 방식을 독립적으로 확장

# 구현부 인터페이스 (데이터 저장 방식)
class DataStorage:
    def save_model(self, model_name, model_data):
        pass
    
    def load_model(self, model_name):
        pass
    
    def save_predictions(self, pred_name, predictions):
        pass

# 구체 구현 (메모리 저장)
class MemoryStorage(DataStorage):
    def __init__(self):
        self.models = {}
        self.predictions = {}
    
    def save_model(self, model_name, model_data):
        self.models[model_name] = model_data
        return f"메모리에 모델 '{model_name}' 저장됨"
    
    def load_model(self, model_name):
        return self.models.get(model_name, None)
    
    def save_predictions(self, pred_name, predictions):
        self.predictions[pred_name] = predictions
        return f"메모리에 예측 결과 '{pred_name}' 저장됨"

# 구체 구현 (파일 저장)
class FileStorage(DataStorage):
    def save_model(self, model_name, model_data):
        # 실제로는 pickle이나 joblib으로 저장
        return f"파일에 모델 '{model_name}' 저장됨 (경로: ./models/{model_name}.pkl)"
    
    def load_model(self, model_name):
        # 실제로는 파일에서 로드
        return f"파일에서 모델 '{model_name}' 로드됨"
    
    def save_predictions(self, pred_name, predictions):
        return f"파일에 예측 결과 '{pred_name}' 저장됨 (경로: ./predictions/{pred_name}.csv)"

# 구체 구현 (데이터베이스 저장)
class DatabaseStorage(DataStorage):
    def save_model(self, model_name, model_data):
        return f"DB에 모델 '{model_name}' 저장됨 (테이블: models)"
    
    def load_model(self, model_name):
        return f"DB에서 모델 '{model_name}' 로드됨"
    
    def save_predictions(self, pred_name, predictions):
        return f"DB에 예측 결과 '{pred_name}' 저장됨 (테이블: predictions)"

# 추상 (ML 모델)
class MLModel:
    def __init__(self, storage):
        self.storage = storage
    
    def train_and_save(self, model_name, X, y):
        pass
    
    def load_and_predict(self, model_name, X, pred_name):
        pass

# 구체 추상 (분류 모델)
class Classifier(MLModel):
    def __init__(self, storage, model_type="rf"):
        super().__init__(storage)
        self.model_type = model_type
    
    def train_and_save(self, model_name, X, y):
        print(f"✓ {self.model_type} 분류 모델 학습 중...")
        model_data = f"trained_{self.model_type}_model"
        result = self.storage.save_model(model_name, model_data)
        return f"모델 학습 및 {result}"
    
    def load_and_predict(self, model_name, X, pred_name):
        loaded_model = self.storage.load_model(model_name)
        print(f"✓ {self.model_type} 모델로 예측 수행 중...")
        predictions = [0, 1, 0, 1, 1]  # 예시 예측 결과
        result = self.storage.save_predictions(pred_name, predictions)
        return f"예측 완료 및 {result}"

# 구체 추상 (회귀 모델) - 새로 추가된 모델 타입
class Regressor(MLModel):
    def __init__(self, storage, model_type="linear"):
        super().__init__(storage)
        self.model_type = model_type
    
    def train_and_save(self, model_name, X, y):
        print(f"✓ {self.model_type} 회귀 모델 학습 중...")
        model_data = f"trained_{self.model_type}_regressor"
        result = self.storage.save_model(model_name, model_data)
        return f"회귀 모델 학습 및 {result}"
    
    def load_and_predict(self, model_name, X, pred_name):
        loaded_model = self.storage.load_model(model_name)
        print(f"✓ {self.model_type} 모델로 회귀 예측 수행 중...")
        predictions = [2.5, 3.1, 1.8, 4.2, 2.9]  # 예시 회귀 예측 결과
        result = self.storage.save_predictions(pred_name, predictions)
        return f"회귀 예측 완료 및 {result}"

# 테스트: 다양한 조합
print("=== 메모리 저장소 + 분류 모델 ===")
memory = MemoryStorage()
classifier_mem = Classifier(memory, "rf")
print(classifier_mem.train_and_save("sentiment_model", "X_train", "y_train"))
print(classifier_mem.load_and_predict("sentiment_model", "X_test", "pred_001"))

print("\n=== 파일 저장소 + 회귀 모델 ===")
file_storage = FileStorage()
regressor_file = Regressor(file_storage, "linear")
print(regressor_file.train_and_save("price_model", "X_train", "y_train"))
print(regressor_file.load_and_predict("price_model", "X_test", "pred_002"))

print("\n=== DB 저장소 + 분류 모델 ===")
db = DatabaseStorage()
classifier_db = Classifier(db, "nn")
print(classifier_db.train_and_save("fraud_model", "X_train", "y_train"))
print(classifier_db.load_and_predict("fraud_model", "X_test", "pred_003"))

print("\n=== 확장성 확인 ===")
# 새로운 모델 타입(군집 모델)을 쉽게 추가할 수 있음
# 새로운 저장 방식(S3, Redis)을 쉽게 추가할 수 있음
# 총 조합: 모델 타입 × 저장 방식

=== 메모리 저장소 + 분류 모델 ===
✓ rf 분류 모델 학습 중...
모델 학습 및 메모리에 모델 'sentiment_model' 저장됨
✓ rf 모델로 예측 수행 중...
예측 완료 및 메모리에 예측 결과 'pred_001' 저장됨

=== 파일 저장소 + 회귀 모델 ===
✓ linear 회귀 모델 학습 중...
회귀 모델 학습 및 파일에 모델 'price_model' 저장됨 (경로: ./models/price_model.pkl)
✓ linear 모델로 회귀 예측 수행 중...
회귀 예측 완료 및 파일에 예측 결과 'pred_002' 저장됨 (경로: ./predictions/pred_002.csv)

=== DB 저장소 + 분류 모델 ===
✓ nn 분류 모델 학습 중...
모델 학습 및 DB에 모델 'fraud_model' 저장됨 (테이블: models)
✓ nn 모델로 예측 수행 중...
예측 완료 및 DB에 예측 결과 'pred_003' 저장됨 (테이블: predictions)

=== 확장성 확인 ===


### 2-3. Composite 패턴

Composite 패턴은 **"트리 구조에서 개별 객체와 그룹 객체를 동일한 인터페이스로 다룰 수 있게 하는 패턴"**입니다.

**비유로 이해하기:**
- **ML 파이프라인**: 개별 전처리기(스케일링)와 복합 전처리기(여러 전처리기 묶음)를 동일하게 "변환기"로 취급
- 개별 변환기: "이 변환기로 데이터 변환"
- 복합 변환기: "이 변환기로 데이터 변환" (내부 변환기들 순차 적용)

**머신러닝 엔지니어 관점:**
ML 파이프라인에서 개별 컴포넌트(전처리기, 특징 선택기)와 복합 컴포넌트(전처리 파이프라인)를 동일한 인터페이스로 다룰 때 유용합니다. 복잡한 파이프라인을 트리 구조로 구성할 수 있습니다.

**코드로 이해하기:**
- `Transformer`: 공통 인터페이스 (transform 메서드)
- `Leaf`: 개별 변환기 (스케일러, 인코더)
- `Composite`: 그룹 변환기 (파이프라인) - 다른 Transformer들을 가질 수 있음

**장점:**
- 클라이언트 코드는 개별/그룹 구분 없이 동일하게 사용
- 새로운 변환기 타입 쉽게 추가 가능
- 파이프라인을 계층적으로 구성 가능

**단점:**
- 모든 컴포넌트가 같은 인터페이스를 가져야 해서 제한적일 수 있음

In [8]:
# Composite 패턴 더 자세한 예제: ML 데이터 변환 파이프라인
# 머신러닝에서 Composite 패턴 적용 예시

class Transformer:
    """데이터 변환기의 기본 인터페이스"""
    def transform(self, data):
        pass
    
    def get_description(self):
        """변환기 설명 반환"""
        pass

class DataPreprocessor(Transformer):
    """개별 전처리기 (Leaf)"""
    def __init__(self, name, transform_func):
        self.name = name
        self.transform_func = transform_func
    
    def transform(self, data):
        print(f"✓ {self.name} 적용 중...")
        return self.transform_func(data)
    
    def get_description(self):
        return f"전처리기: {self.name}"

class PreprocessingPipeline(Transformer):
    """전처리 파이프라인 (Composite)"""
    def __init__(self, name="Pipeline"):
        self.name = name
        self.transformers = []
    
    def add_transformer(self, transformer):
        self.transformers.append(transformer)
    
    def transform(self, data):
        result = data
        for transformer in self.transformers:
            result = transformer.transform(result)
        return result
    
    def get_description(self):
        descriptions = [t.get_description() for t in self.transformers]
        return f"파이프라인 '{self.name}': {descriptions}"

# 테스트: ML 전처리 파이프라인 구성
print("=== ML 전처리 파이프라인 구성 ===")

# 개별 전처리기들 (Leaf)
scaler = DataPreprocessor(
    "StandardScaler",
    lambda data: f"표준화된 {data}"
)

encoder = DataPreprocessor(
    "OneHotEncoder", 
    lambda data: f"원핫인코딩된 {data}"
)

imputer = DataPreprocessor(
    "SimpleImputer",
    lambda data: f"결측치처리된 {data}"
)

print(f"개별 변환기들:")
print(f"- {scaler.get_description()}")
print(f"- {encoder.get_description()}")
print(f"- {imputer.get_description()}")

# 기본 전처리 파이프라인 (Composite)
basic_pipeline = PreprocessingPipeline("기본 전처리")
basic_pipeline.add_transformer(imputer)
basic_pipeline.add_transformer(scaler)

print(f"\n{basic_pipeline.get_description()}")

# 고급 전처리 파이프라인 (Composite)
advanced_pipeline = PreprocessingPipeline("고급 전처리")
advanced_pipeline.add_transformer(imputer)
advanced_pipeline.add_transformer(encoder)
advanced_pipeline.add_transformer(scaler)

print(f"\n{advanced_pipeline.get_description()}")

# 파이프라인 실행
raw_data = "원본 데이터"
print(f"\n입력 데이터: {raw_data}")

processed_basic = basic_pipeline.transform(raw_data)
print(f"기본 파이프라인 결과: {processed_basic}")

processed_advanced = advanced_pipeline.transform(raw_data)
print(f"고급 파이프라인 결과: {processed_advanced}")

# 파이프라인 중첩 (Composite 안에 Composite)
full_pipeline = PreprocessingPipeline("전체 파이프라인")
full_pipeline.add_transformer(basic_pipeline)  # 파이프라인을 컴포넌트로 추가
full_pipeline.add_transformer(encoder)

print(f"\n{full_pipeline.get_description()}")
final_result = full_pipeline.transform(raw_data)
print(f"전체 파이프라인 결과: {final_result}")

print("\n🎯 Composite 패턴으로 개별 변환기와 파이프라인을 동일하게 다룰 수 있음!")
print("   복잡한 전처리 워크플로우를 계층적으로 구성 가능")

=== ML 전처리 파이프라인 구성 ===
개별 변환기들:
- 전처리기: StandardScaler
- 전처리기: OneHotEncoder
- 전처리기: SimpleImputer

파이프라인 '기본 전처리': ['전처리기: SimpleImputer', '전처리기: StandardScaler']

파이프라인 '고급 전처리': ['전처리기: SimpleImputer', '전처리기: OneHotEncoder', '전처리기: StandardScaler']

입력 데이터: 원본 데이터
✓ SimpleImputer 적용 중...
✓ StandardScaler 적용 중...
기본 파이프라인 결과: 표준화된 결측치처리된 원본 데이터
✓ SimpleImputer 적용 중...
✓ OneHotEncoder 적용 중...
✓ StandardScaler 적용 중...
고급 파이프라인 결과: 표준화된 원핫인코딩된 결측치처리된 원본 데이터

파이프라인 '전체 파이프라인': ["파이프라인 '기본 전처리': ['전처리기: SimpleImputer', '전처리기: StandardScaler']", '전처리기: OneHotEncoder']
✓ SimpleImputer 적용 중...
✓ StandardScaler 적용 중...
✓ OneHotEncoder 적용 중...
전체 파이프라인 결과: 원핫인코딩된 표준화된 결측치처리된 원본 데이터

🎯 Composite 패턴으로 개별 변환기와 파이프라인을 동일하게 다룰 수 있음!
   복잡한 전처리 워크플로우를 계층적으로 구성 가능


### 2-4. Decorator 패턴

**목적**: 객체에 기능을 동적으로 추가하는 방법

**사용 시기**:
- Runtime에 기능을 추가하거나 제거하고 싶을 때
- 기능 조합이 자유로워야 할 때

**장점**: 상속을 많이 쓰지 않고 유연하게 확장 가능

**단점**: 데코레이터가 많이 중첩되면 에러 추적이 어려움

### 2-4. Decorator 패턴

Decorator 패턴은 **"객체에 기능을 동적으로 추가하는 방법"**입니다. 기존 객체를 수정하지 않고 런타임에 새로운 기능을 덧붙일 수 있습니다.

**비유로 이해하기:**
- **커피 주문**: 기본 커피에 우유, 설탕, 휘핑크림 등을 추가 주문
- 각 추가는 기존 커피를 감싸서 새로운 기능을 더함
- 최종 가격과 설명이 동적으로 계산됨

**코드로 이해하기:**
- `Component`: 기본 인터페이스 (커피)
- `ConcreteComponent`: 실제 객체 (기본 커피)
- `Decorator`: 감싸는 추상 클래스
- `ConcreteDecorator`: 구체적인 추가 기능 (우유, 설탕)

**머신러닝 엔지니어 관점:**
머신러닝에서는 모델이나 데이터 파이프라인에 로깅, 캐싱, 타이밍, 검증 등의 기능을 동적으로 추가할 때 유용합니다. 예를 들어, 모델 예측 함수에 실행 시간 측정이나 결과 로깅을 추가할 수 있습니다.

**장점:**
- 상속 없이 유연하게 기능 확장
- 런타임에 조합 가능 (다중 상속의 단점 피함)
- OCP(개방-폐쇄 원칙) 준수

**단점:**
- 데코레이터가 많아지면 객체 구조가 복잡해짐
- 디버깅 시 호출 스택이 깊어짐
- 초기 설계가 중요 (인터페이스 일관성)

In [9]:
# Decorator 패턴 더 자세한 예제: 머신러닝 모델에 기능 추가
# 머신러닝에서 Decorator 패턴 적용 예시

from abc import ABC, abstractmethod
import time
from functools import wraps

# 기본 모델 인터페이스
class MLModel(ABC):
    @abstractmethod
    def predict(self, X):
        pass

# 구체 모델: 간단한 선형 회귀 (시뮬레이션)
class LinearRegressionModel(MLModel):
    def __init__(self):
        self.weights = [0.1, 0.2, 0.3]  # 가상의 가중치

    def predict(self, X):
        # 간단한 예측: 가중치와 입력의 내적
        return sum(w * x for w, x in zip(self.weights, X))

# 데코레이터 추상 클래스
class ModelDecorator(MLModel):
    def __init__(self, model):
        self._model = model

    def predict(self, X):
        return self._model.predict(X)

# 구체 데코레이터: 실행 시간 측정
class TimingDecorator(ModelDecorator):
    def predict(self, X):
        start_time = time.time()
        result = self._model.predict(X)
        end_time = time.time()
        print(f"예측 시간: {end_time - start_time:.4f}초")
        return result

# 구체 데코레이터: 결과 로깅
class LoggingDecorator(ModelDecorator):
    def predict(self, X):
        result = self._model.predict(X)
        print(f"입력: {X}, 예측 결과: {result:.2f}")
        return result

# 구체 데코레이터: 입력 검증
class ValidationDecorator(ModelDecorator):
    def predict(self, X):
        if not isinstance(X, list) or len(X) != 3:
            raise ValueError("입력은 3개의 숫자 리스트여야 합니다")
        return self._model.predict(X)

# 테스트: 데코레이터 조합
print("=== 기본 모델 ===")
base_model = LinearRegressionModel()
test_input = [1.0, 2.0, 3.0]
print(f"기본 예측: {base_model.predict(test_input):.2f}")

print("\n=== 타이밍 데코레이터 추가 ===")
timed_model = TimingDecorator(base_model)
print(f"타이밍 예측: {timed_model.predict(test_input):.2f}")

print("\n=== 로깅 + 타이밍 데코레이터 ===")
logged_timed_model = LoggingDecorator(TimingDecorator(base_model))
print(f"로깅+타이밍 예측: {logged_timed_model.predict(test_input):.2f}")

print("\n=== 검증 + 로깅 + 타이밍 데코레이터 ===")
full_model = ValidationDecorator(LoggingDecorator(TimingDecorator(base_model)))
try:
    print(f"풀 데코레이터 예측: {full_model.predict(test_input):.2f}")
except ValueError as e:
    print(f"에러: {e}")

print("\n🎯 Decorator 패턴으로 모델에 기능을 유연하게 추가할 수 있음!")
print("   각 데코레이터는 독립적으로 조합 가능")

=== 기본 모델 ===
기본 예측: 1.40

=== 타이밍 데코레이터 추가 ===
예측 시간: 0.0000초
타이밍 예측: 1.40

=== 로깅 + 타이밍 데코레이터 ===
예측 시간: 0.0000초
입력: [1.0, 2.0, 3.0], 예측 결과: 1.40
로깅+타이밍 예측: 1.40

=== 검증 + 로깅 + 타이밍 데코레이터 ===
예측 시간: 0.0000초
입력: [1.0, 2.0, 3.0], 예측 결과: 1.40
풀 데코레이터 예측: 1.40

🎯 Decorator 패턴으로 모델에 기능을 유연하게 추가할 수 있음!
   각 데코레이터는 독립적으로 조합 가능


### 2-5. Facade 패턴

Facade 패턴은 **"복잡한 서브시스템들을 단일 인터페이스로 감싸 외부에서 쉬운 사용법으로 제공하는 패턴"**입니다.

**비유로 이해하기:**
- **자동차 운전**: 엔진, 변속기, 브레이크 등 복잡한 부품들을 "핸들, 액셀, 브레이크"라는 단순한 인터페이스로 제공
- 사용자는 내부 복잡성을 모르고도 차를 운전할 수 있음

**코드로 이해하기:**
- `Subsystem`: 복잡한 내부 클래스들 (데이터 로딩, 전처리, 모델 학습 등)
- `Facade`: 단순한 인터페이스 제공 (train_model() 같은 메서드)

**머신러닝 엔지니어 관점:**
머신러닝 프로젝트에서 데이터 로딩, 전처리, 모델 학습, 평가, 배포까지의 복잡한 파이프라인을 하나의 Facade 클래스로 감싸서, 다른 팀원들이 쉽게 사용할 수 있게 합니다. 예를 들어, "model.train()" 하나로 모든 복잡한 과정을 처리할 수 있습니다.

**장점:**
- 사용법이 단순해지고 의존성이 축소됨
- 서브시스템 변경이 Facade에만 영향
- 코드 재사용성 향상

**단점:**
- Facade가 모든 기능을 감싸면 비대해질 수 있음
- 너무 많은 Facade는 오히려 복잡성을 증가시킬 수 있음

In [10]:
# Facade 패턴 더 자세한 예제: 머신러닝 파이프라인 단순화
# 머신러닝에서 Facade 패턴 적용 예시

# 서브시스템: 복잡한 ML 파이프라인 구성 요소들
class DataLoader:
    """데이터 로딩 서브시스템"""
    def load_data(self, file_path):
        print(f"데이터 파일 로드: {file_path}")
        # 실제로는 pandas.read_csv() 등 사용
        return f"데이터셋 ({file_path})"

class DataPreprocessor:
    """데이터 전처리 서브시스템"""
    def preprocess(self, data):
        print("데이터 전처리: 결측치 처리, 스케일링, 인코딩")
        return f"전처리된 {data}"

class ModelTrainer:
    """모델 학습 서브시스템"""
    def train(self, processed_data, model_type="random_forest"):
        print(f"모델 학습: {model_type} 알고리즘 사용")
        print("하이퍼파라미터 튜닝, 교차 검증 수행")
        return f"학습된 {model_type} 모델"

class ModelEvaluator:
    """모델 평가 서브시스템"""
    def evaluate(self, model, test_data):
        print("모델 평가: 정확도, 정밀도, 재현율 계산")
        print("ROC 곡선, 혼동 행렬 생성")
        return {"accuracy": 0.85, "precision": 0.82, "recall": 0.88}

# Facade: 복잡한 파이프라인을 단순한 인터페이스로 제공
class MLTrainingFacade:
    def __init__(self):
        self.loader = DataLoader()
        self.preprocessor = DataPreprocessor()
        self.trainer = ModelTrainer()
        self.evaluator = ModelEvaluator()

    def train_model(self, data_path, model_type="random_forest"):
        """단순한 인터페이스: 데이터 경로만 주면 전체 파이프라인 실행"""
        print("=== ML 파이프라인 시작 ===")
        
        # 데이터 로딩
        raw_data = self.loader.load_data(data_path)
        
        # 전처리
        processed_data = self.preprocessor.preprocess(raw_data)
        
        # 모델 학습
        model = self.trainer.train(processed_data, model_type)
        
        # 평가 (간단한 테스트 데이터로 가정)
        metrics = self.evaluator.evaluate(model, "test_data")
        
        print("=== ML 파이프라인 완료 ===")
        return {
            "model": model,
            "metrics": metrics
        }

# 테스트: Facade를 통한 단순한 사용
print("=== 데이터 사이언티스트의 사용법 ===")
facade = MLTrainingFacade()

# 복잡한 내부 과정을 모르고도 쉽게 사용 가능
result = facade.train_model("customer_data.csv", model_type="xgboost")

print(f"\n최종 결과: {result['model']}")
print(f"평가 지표: {result['metrics']}")

print("\n🎯 Facade 패턴으로 복잡한 ML 파이프라인을 단순한 API로 제공!")
print("   내부 구현 변경이 외부 사용자에게 영향 없음")

=== 데이터 사이언티스트의 사용법 ===
=== ML 파이프라인 시작 ===
데이터 파일 로드: customer_data.csv
데이터 전처리: 결측치 처리, 스케일링, 인코딩
모델 학습: xgboost 알고리즘 사용
하이퍼파라미터 튜닝, 교차 검증 수행
모델 평가: 정확도, 정밀도, 재현율 계산
ROC 곡선, 혼동 행렬 생성
=== ML 파이프라인 완료 ===

최종 결과: 학습된 xgboost 모델
평가 지표: {'accuracy': 0.85, 'precision': 0.82, 'recall': 0.88}

🎯 Facade 패턴으로 복잡한 ML 파이프라인을 단순한 API로 제공!
   내부 구현 변경이 외부 사용자에게 영향 없음


### 2-6. Flyweight 패턴

Flyweight 패턴은 **"많은 수의 유사 객체들을 메모리 효율적으로 만들기 위해 내부 상태를 공유하는 패턴"**입니다.

**비유로 이해하기:**
- **문자열 풀**: 같은 문자열("hello")을 여러 변수가 공유
- 각 변수는 고유한 메모리 주소지만, 실제 데이터는 하나만 저장
- 메모리 사용량이 크게 줄어듦

**코드로 이해하기:**
- `Flyweight`: 공유되는 객체 (내부 상태만 가짐)
- `FlyweightFactory`: Flyweight 객체들을 관리하고 재사용
- 내부 상태(intrinsic): 공유 가능 (예: 단어 임베딩 벡터)
- 외부 상태(extrinsic): 공유 불가 (예: 문장에서의 위치)

**머신러닝 엔지니어 관점:**
대용량 텍스트 데이터나 이미지 데이터에서 동일한 단어나 특성 벡터가 반복될 때 메모리를 절약합니다. 예를 들어, 수백만 개의 문장에서 같은 단어("the")의 임베딩 벡터를 공유하면 메모리 사용이 크게 줄어듭니다.

**장점:**
- 메모리 사용량 대폭 감소
- 많은 객체를 효율적으로 관리
- 성능 향상 (특히 메모리 제한 환경)

**단점:**
- 설계 복잡성 증가
- 외부 상태 관리가 번거로움
- 디버깅 어려움

In [11]:
# Flyweight 패턴 더 자세한 예제: 단어 임베딩 공유
# 머신러닝에서 Flyweight 패턴 적용 예시

import numpy as np

class WordEmbedding:
    """Flyweight: 공유되는 단어 임베딩 벡터"""
    def __init__(self, word, vector):
        self.word = word  # 내부 상태: 공유 가능
        self.vector = vector  # 내부 상태: 공유 가능

    def get_similarity(self, other_embedding, position_in_text):
        """외부 상태(position_in_text)를 고려한 유사도 계산"""
        # 코사인 유사도 계산 (간단한 버전)
        dot_product = np.dot(self.vector, other_embedding.vector)
        norm_a = np.linalg.norm(self.vector)
        norm_b = np.linalg.norm(other_embedding.vector)
        similarity = dot_product / (norm_a * norm_b)
        
        # 위치에 따른 가중치 적용 (외부 상태 사용)
        position_weight = 1.0 / (1.0 + abs(position_in_text))  # 가까운 단어일수록 가중치 높음
        return similarity * position_weight

class EmbeddingFactory:
    """FlyweightFactory: 임베딩 객체들을 관리하고 재사용"""
    def __init__(self):
        self._embeddings = {}

    def get_embedding(self, word):
        if word not in self._embeddings:
            # 실제로는 GloVe, Word2Vec 등에서 로드
            # 여기서는 랜덤 벡터로 시뮬레이션
            vector = np.random.rand(50)  # 50차원 임베딩
            self._embeddings[word] = WordEmbedding(word, vector)
            print(f"새로운 임베딩 생성: {word}")
        else:
            print(f"기존 임베딩 재사용: {word}")
        return self._embeddings[word]

# 테스트: 대용량 텍스트 데이터에서 단어 임베딩 공유
print("=== 단어 임베딩 Flyweight 테스트 ===")

factory = EmbeddingFactory()

# 같은 단어가 여러 번 나타나는 텍스트 시뮬레이션
words_in_text = ["the", "cat", "sat", "on", "the", "mat", "the", "cat", "slept"]

embeddings = []
for i, word in enumerate(words_in_text):
    emb = factory.get_embedding(word)
    embeddings.append((emb, i))  # (임베딩, 텍스트 내 위치)

print(f"\n총 단어 수: {len(words_in_text)}")
print(f"고유 단어 수: {len(factory._embeddings)}")
print(f"메모리 절약: {(len(words_in_text) - len(factory._embeddings)) / len(words_in_text) * 100:.1f}%")

# 유사도 계산 예시 (외부 상태 사용)
print("\n=== 단어 유사도 계산 (위치 고려) ===")
cat_emb = factory.get_embedding("cat")
the_emb = factory.get_embedding("the")

# 같은 단어의 다른 위치에서의 유사도
similarity1 = cat_emb.get_similarity(the_emb, position_in_text=1)  # 가까운 위치
similarity2 = cat_emb.get_similarity(the_emb, position_in_text=10)  # 먼 위치

print(f"'cat'과 'the'의 유사도 (위치 1): {similarity1:.3f}")
print(f"'cat'과 'the'의 유사도 (위치 10): {similarity2:.3f}")

print("\n🎯 Flyweight 패턴으로 대용량 텍스트 데이터의 메모리 사용 최적화!")
print("   동일한 단어 임베딩을 공유하여 메모리 절약")

=== 단어 임베딩 Flyweight 테스트 ===
새로운 임베딩 생성: the
새로운 임베딩 생성: cat
새로운 임베딩 생성: sat
새로운 임베딩 생성: on
기존 임베딩 재사용: the
새로운 임베딩 생성: mat
기존 임베딩 재사용: the
기존 임베딩 재사용: cat
새로운 임베딩 생성: slept

총 단어 수: 9
고유 단어 수: 6
메모리 절약: 33.3%

=== 단어 유사도 계산 (위치 고려) ===
기존 임베딩 재사용: cat
기존 임베딩 재사용: the
'cat'과 'the'의 유사도 (위치 1): 0.401
'cat'과 'the'의 유사도 (위치 10): 0.073

🎯 Flyweight 패턴으로 대용량 텍스트 데이터의 메모리 사용 최적화!
   동일한 단어 임베딩을 공유하여 메모리 절약


### 2-7. Proxy 패턴

Proxy 패턴은 **"실제 객체에 접근할 때 중간에서 대리(프록시)를 두어 접근 제어나 추가 작업을 수행하는 패턴"**입니다.

**비유로 이해하기:**
- **신용카드**: 실제 돈 대신 카드로 결제
- 은행은 카드 사용을 검증하고 기록
- 실제 돈 이동은 나중에 일괄 처리

**코드로 이해하기:**
- `Subject`: 실제 객체와 프록시의 공통 인터페이스
- `RealSubject`: 실제 객체 (무거운 모델)
- `Proxy`: 프록시 객체 (접근 제어, 캐싱, 로깅)

**머신러닝 엔지니어 관점:**
대용량 모델을 메모리에 로드하는 비용이 클 때 유용합니다. 모델을 처음 사용할 때만 로드하고, 이후에는 캐시된 결과를 반환하거나, 사용자 권한을 확인한 후에만 예측을 허용합니다.

**장점:**
- 실제 객체를 수정하지 않고 기능 추가
- 지연 로딩으로 성능 향상
- 접근 제어 및 보안 강화

**단점:**
- 프록시 계층이 증가하면 구조 복잡
- 간단한 경우 오버헤드만 증가할 수 있음

In [12]:
# Proxy 패턴 더 자세한 예제: 모델 예측 프록시
# 머신러닝에서 Proxy 패턴 적용 예시

import time
import hashlib

# Subject: 공통 인터페이스
class MLModel:
    def predict(self, input_data):
        pass

# RealSubject: 실제 대용량 모델
class RealMLModel(MLModel):
    def __init__(self, model_name):
        self.model_name = model_name
        self._load_model()  # 무거운 작업

    def _load_model(self):
        print(f"대용량 모델 로드 중: {self.model_name}...")
        time.sleep(2)  # 모델 로딩 시뮬레이션
        print(f"모델 로드 완료: {self.model_name}")
        self.is_loaded = True

    def predict(self, input_data):
        if not self.is_loaded:
            raise RuntimeError("모델이 로드되지 않았습니다")
        
        # 실제 예측 로직 (시뮬레이션)
        input_hash = hashlib.md5(str(input_data).encode()).hexdigest()[:8]
        prediction = float(int(input_hash, 16) % 100) / 100.0
        return f"{self.model_name} 예측 결과: {prediction:.3f}"

# Proxy: 모델 접근 제어 및 최적화
class ModelProxy(MLModel):
    def __init__(self, model_name, user_role="user"):
        self.model_name = model_name
        self.user_role = user_role
        self._real_model = None
        self._cache = {}  # 예측 결과 캐시

    def _check_access(self):
        """접근 권한 확인"""
        allowed_roles = ["admin", "ml_engineer", "data_scientist"]
        if self.user_role not in allowed_roles:
            raise PermissionError(f"접근 권한 없음. 역할: {self.user_role}")
        return True

    def _get_cache_key(self, input_data):
        """캐시 키 생성"""
        return hashlib.md5(str(input_data).encode()).hexdigest()

    def predict(self, input_data):
        # 1. 접근 권한 확인
        self._check_access()
        
        # 2. 캐시 확인
        cache_key = self._get_cache_key(input_data)
        if cache_key in self._cache:
            print("캐시된 결과 반환")
            return f"[캐시] {self._cache[cache_key]}"
        
        # 3. 실제 모델이 없으면 지연 로딩
        if self._real_model is None:
            print("실제 모델을 처음 요청하여 로드...")
            self._real_model = RealMLModel(self.model_name)
        
        # 4. 예측 수행 및 캐시 저장
        result = self._real_model.predict(input_data)
        self._cache[cache_key] = result
        
        return result

# 테스트: 프록시를 통한 모델 사용
print("=== 일반 사용자 (접근 거부) ===")
try:
    proxy_user = ModelProxy("sentiment_model", user_role="user")
    proxy_user.predict([1, 2, 3])
except PermissionError as e:
    print(f"에러: {e}")

print("\n=== 데이터 사이언티스트 (접근 허용) ===")
proxy_ds = ModelProxy("sentiment_model", user_role="data_scientist")

# 첫 번째 예측: 모델 로드
print("첫 번째 예측:")
result1 = proxy_ds.predict([1, 2, 3])
print(result1)

# 두 번째 예측: 같은 입력으로 캐시 사용
print("\n두 번째 예측 (같은 입력):")
result2 = proxy_ds.predict([1, 2, 3])
print(result2)

# 세 번째 예측: 다른 입력
print("\n세 번째 예측 (다른 입력):")
result3 = proxy_ds.predict([4, 5, 6])
print(result3)

print("\n🎯 Proxy 패턴으로 모델 로딩 최적화 및 접근 제어!")
print("   지연 로딩 + 캐싱 + 권한 확인")

=== 일반 사용자 (접근 거부) ===
에러: 접근 권한 없음. 역할: user

=== 데이터 사이언티스트 (접근 허용) ===
첫 번째 예측:
실제 모델을 처음 요청하여 로드...
대용량 모델 로드 중: sentiment_model...
모델 로드 완료: sentiment_model
sentiment_model 예측 결과: 0.680

두 번째 예측 (같은 입력):
캐시된 결과 반환
[캐시] sentiment_model 예측 결과: 0.680

세 번째 예측 (다른 입력):
sentiment_model 예측 결과: 0.830

🎯 Proxy 패턴으로 모델 로딩 최적화 및 접근 제어!
   지연 로딩 + 캐싱 + 권한 확인


---
## 3. 행동 (Behavioral) 패턴

행동 패턴은 "객체들이 어떻게 협력하고 행동을 바꾸는지" 다루는 설계법입니다.

## 3. 행동 (Behavioral) 패턴

행동 패턴은 "객체들이 어떻게 협력하고 행동을 바꾸는지" 다루는 설계법입니다. 머신러닝에서는 데이터 파이프라인, 모델 선택, 실험 관리 등에 자주 사용됩니다.

### 3-1. Chain of Responsibility 패턴

Chain of Responsibility 패턴은 **"요청을 처리할 가능성이 있는 여러 객체를 체인으로 이어 각 객체가 요청을 처리하거나 다음 객체로 넘기는 패턴"**입니다.

**비유로 이해하기:**
- **고객 지원**: 1차 지원 → 2차 지원 → 관리자 → CEO
- 각 단계에서 해결 가능하면 처리, 아니면 상위로 이관

**코드로 이해하기:**
- `Handler`: 공통 인터페이스 (handle_request 메서드)
- `ConcreteHandler`: 구체적인 처리자 (데이터 검증, 전처리 등)
- 체인 연결: 각 핸들러가 다음 핸들러를 가리킴

**머신러닝 엔지니어 관점:**
ML 파이프라인에서 데이터가 여러 단계의 검증과 처리를 거쳐야 할 때 유용합니다. 예를 들어, 원본 데이터 → 데이터 검증 → 결측치 처리 → 이상치 제거 → 스케일링 → 모델 예측.

**장점:**
- 처리 책임을 유연하게 분산
- 새로운 처리 단계 쉽게 추가
- 결합도 낮음

**단점:**
- 처리가 누락될 수 있음
- 디버깅 어려움 (체인 전체를 추적해야 함)

In [13]:
# Chain of Responsibility 패턴 더 자세한 예제: ML 데이터 전처리 파이프라인
# 머신러닝에서 Chain of Responsibility 패턴 적용 예시

import pandas as pd
import numpy as np

class DataHandler:
    """기본 핸들러 클래스"""
    def __init__(self, successor=None):
        self.successor = successor
    
    def handle(self, data):
        """데이터 처리 또는 다음 핸들러로 전달"""
        if self.successor:
            return self.successor.handle(data)
        return data

class DataValidationHandler(DataHandler):
    """데이터 검증 핸들러"""
    def handle(self, data):
        print("1. 데이터 검증 중...")
        if not isinstance(data, pd.DataFrame):
            raise ValueError("입력은 pandas DataFrame이어야 합니다")
        
        required_cols = ['feature1', 'feature2', 'target']
        missing_cols = [col for col in required_cols if col not in data.columns]
        if missing_cols:
            raise ValueError(f"필수 컬럼 누락: {missing_cols}")
        
        print("✓ 데이터 검증 통과")
        return super().handle(data)

class MissingValueHandler(DataHandler):
    """결측치 처리 핸들러"""
    def handle(self, data):
        print("2. 결측치 처리 중...")
        missing_before = data.isnull().sum().sum()
        
        # 간단한 결측치 처리: 평균으로 채움
        numeric_cols = data.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if data[col].isnull().any():
                data[col].fillna(data[col].mean(), inplace=True)
        
        missing_after = data.isnull().sum().sum()
        print(f"✓ 결측치 처리 완료: {missing_before} → {missing_after}")
        return super().handle(data)

class OutlierHandler(DataHandler):
    """이상치 처리 핸들러"""
    def handle(self, data):
        print("3. 이상치 처리 중...")
        numeric_cols = data.select_dtypes(include=[np.number]).columns
        
        for col in numeric_cols:
            if col == 'target':  # 타겟 변수는 이상치 처리하지 않음
                continue
                
            # IQR 방법으로 이상치 제거
            Q1 = data[col].quantile(0.25)
            Q3 = data[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            outliers = ((data[col] < lower_bound) | (data[col] > upper_bound)).sum()
            data = data[(data[col] >= lower_bound) & (data[col] <= upper_bound)]
            
            if outliers > 0:
                print(f"✓ {col} 컬럼에서 이상치 {outliers}개 제거")
        
        print(f"✓ 최종 데이터 크기: {data.shape}")
        return super().handle(data)

class ScalingHandler(DataHandler):
    """스케일링 핸들러"""
    def handle(self, data):
        print("4. 데이터 스케일링 중...")
        numeric_cols = data.select_dtypes(include=[np.number]).columns
        
        for col in numeric_cols:
            if col == 'target':  # 타겟 변수는 스케일링하지 않음
                continue
            # Min-Max 스케일링
            min_val, max_val = data[col].min(), data[col].max()
            data[col] = (data[col] - min_val) / (max_val - min_val)
            print(f"✓ {col} 스케일링 완료: [{min_val:.2f}, {max_val:.2f}] → [0, 1]")
        
        print("✓ 스케일링 완료")
        return super().handle(data)

# 테스트: 전처리 체인 구성 및 실행
print("=== ML 데이터 전처리 체인 테스트 ===")

# 샘플 데이터 생성 (결측치와 이상치 포함)
np.random.seed(42)
data = pd.DataFrame({
    'feature1': [1, 2, np.nan, 4, 100],  # 결측치와 이상치 포함
    'feature2': [10, 20, 30, np.nan, 50],
    'target': [0, 1, 0, 1, 0]
})

print("원본 데이터:")
print(data)
print(f"원본 데이터 shape: {data.shape}")

# 전처리 체인 구성: 검증 → 결측치 → 이상치 → 스케일링
processing_chain = DataValidationHandler(
    MissingValueHandler(
        OutlierHandler(
            ScalingHandler()
        )
    )
)

try:
    # 체인 실행
    processed_data = processing_chain.handle(data.copy())
    
    print("\n처리된 데이터:")
    print(processed_data)
    print(f"처리된 데이터 shape: {processed_data.shape}")
    
except Exception as e:
    print(f"전처리 중 에러 발생: {e}")

print("\n🎯 Chain of Responsibility로 유연한 데이터 전처리 파이프라인 구축!")
print("   각 단계 독립적으로 추가/제거/수정 가능")

=== ML 데이터 전처리 체인 테스트 ===
원본 데이터:
   feature1  feature2  target
0       1.0      10.0       0
1       2.0      20.0       1
2       NaN      30.0       0
3       4.0       NaN       1
4     100.0      50.0       0
원본 데이터 shape: (5, 3)
1. 데이터 검증 중...
✓ 데이터 검증 통과
2. 결측치 처리 중...
✓ 결측치 처리 완료: 2 → 0
3. 이상치 처리 중...
✓ feature1 컬럼에서 이상치 1개 제거
✓ 최종 데이터 크기: (4, 3)
4. 데이터 스케일링 중...
✓ feature1 스케일링 완료: [1.00, 26.75] → [0, 1]
✓ feature2 스케일링 완료: [10.00, 30.00] → [0, 1]
✓ 스케일링 완료

처리된 데이터:
   feature1  feature2  target
0  0.000000     0.000       0
1  0.038835     0.500       1
2  1.000000     1.000       0
3  0.116505     0.875       1
처리된 데이터 shape: (4, 3)

🎯 Chain of Responsibility로 유연한 데이터 전처리 파이프라인 구축!
   각 단계 독립적으로 추가/제거/수정 가능


### 3-2. Command 패턴

**목적**: 요청(행위)을 객체(명령)로 캡슐화하여 호출자와 수행자 사이를 분리

**사용 시기**:
- 행동 취소(undo), 작업 기록, 작업 큐 등

**장점**: 요청을 객체로 다루어 유연하게 저장·전달·취소 가능

**단점**: 명령 클래스가 늘어나 코드가 많아질 수 있음

### 3-2. Command 패턴

Command 패턴은 **"요청(행위)을 객체(명령)로 캡슐화하여 호출자와 수행자 사이를 분리하는 패턴"**입니다.

**비유로 이해하기:**
- **리모컨 버튼**: 버튼을 누르면 TV가 켜지지만, 버튼은 TV의 내부를 모름
- 각 버튼은 특정 명령을 캡슐화
- undo 기능으로 이전 상태로 되돌릴 수 있음

**코드로 이해하기:**
- `Command`: 명령 인터페이스 (execute, undo 메서드)
- `ConcreteCommand`: 구체적인 명령 (모델 학습, 데이터 분석 등)
- `Invoker`: 명령을 호출하는 객체 (실험 관리자)
- `Receiver`: 명령을 실제로 수행하는 객체 (ML 파이프라인)

**머신러닝 엔지니어 관점:**
ML 실험에서 작업을 객체로 만들어 저장하고 재실행할 수 있습니다. 예를 들어, "모델 학습" 명령을 저장했다가 나중에 undo하거나 다른 환경에서 재실행할 수 있습니다.

**장점:**
- 작업을 객체로 다루어 유연하게 저장·전달·취소 가능
- 새로운 명령 쉽게 추가
- 작업 기록 및 재현성 향상

**단점:**
- 명령 클래스가 많아질 수 있음
- 간단한 작업에 오버헤드

In [14]:
# Command 패턴 더 자세한 예제: ML 실험 관리 시스템
# 머신러닝에서 Command 패턴 적용 예시

from abc import ABC, abstractmethod
import time

# Command 인터페이스
class MLCommand(ABC):
    @abstractmethod
    def execute(self):
        pass
    
    @abstractmethod
    def undo(self):
        pass

# Receiver: ML 파이프라인 (명령을 실제로 수행)
class MLPipeline:
    def __init__(self):
        self.models = {}
        self.deployed_model = None
    
    def train_model(self, model_name, algorithm, params):
        print(f"모델 학습 시작: {model_name} ({algorithm})")
        time.sleep(1)  # 학습 시뮬레이션
        self.models[model_name] = {
            'algorithm': algorithm,
            'params': params,
            'trained_at': time.time(),
            'status': 'trained'
        }
        print(f"✓ 모델 학습 완료: {model_name}")
        return self.models[model_name]
    
    def evaluate_model(self, model_name):
        if model_name not in self.models:
            raise ValueError(f"모델 {model_name}이 존재하지 않습니다")
        
        print(f"모델 평가 시작: {model_name}")
        time.sleep(0.5)  # 평가 시뮬레이션
        # 가상의 평가 결과
        metrics = {
            'accuracy': 0.85 + np.random.random() * 0.1,
            'precision': 0.82 + np.random.random() * 0.1,
            'recall': 0.88 + np.random.random() * 0.1
        }
        self.models[model_name]['metrics'] = metrics
        print(f"✓ 모델 평가 완료: {model_name}")
        return metrics
    
    def deploy_model(self, model_name):
        if model_name not in self.models:
            raise ValueError(f"모델 {model_name}이 존재하지 않습니다")
        
        print(f"모델 배포 시작: {model_name}")
        time.sleep(0.5)  # 배포 시뮬레이션
        old_deployed = self.deployed_model
        self.deployed_model = model_name
        self.models[model_name]['status'] = 'deployed'
        print(f"✓ 모델 배포 완료: {model_name}")
        return old_deployed
    
    def undeploy_model(self, model_name):
        if self.deployed_model == model_name:
            print(f"모델 배포 취소: {model_name}")
            self.deployed_model = None
            self.models[model_name]['status'] = 'trained'
        else:
            print(f"모델 {model_name}은 배포되지 않았습니다")

# Concrete Commands
class TrainModelCommand(MLCommand):
    def __init__(self, pipeline, model_name, algorithm, params):
        self.pipeline = pipeline
        self.model_name = model_name
        self.algorithm = algorithm
        self.params = params
        self.backup = None
    
    def execute(self):
        self.backup = self.pipeline.models.get(self.model_name)
        return self.pipeline.train_model(self.model_name, self.algorithm, self.params)
    
    def undo(self):
        if self.backup:
            self.pipeline.models[self.model_name] = self.backup
        else:
            self.pipeline.models.pop(self.model_name, None)
        print(f"모델 학습 취소: {self.model_name}")

class EvaluateModelCommand(MLCommand):
    def __init__(self, pipeline, model_name):
        self.pipeline = pipeline
        self.model_name = model_name
        self.backup_metrics = None
    
    def execute(self):
        self.backup_metrics = self.pipeline.models.get(self.model_name, {}).get('metrics')
        return self.pipeline.evaluate_model(self.model_name)
    
    def undo(self):
        if self.backup_metrics:
            self.pipeline.models[self.model_name]['metrics'] = self.backup_metrics
        else:
            self.pipeline.models[self.model_name].pop('metrics', None)
        print(f"모델 평가 취소: {self.model_name}")

class DeployModelCommand(MLCommand):
    def __init__(self, pipeline, model_name):
        self.pipeline = pipeline
        self.model_name = model_name
        self.previous_deployed = None
    
    def execute(self):
        self.previous_deployed = self.pipeline.deploy_model(self.model_name)
        return self.model_name
    
    def undo(self):
        self.pipeline.undeploy_model(self.model_name)
        if self.previous_deployed:
            self.pipeline.deploy_model(self.previous_deployed)

# Invoker: 실험 관리자
class ExperimentManager:
    def __init__(self):
        self.pipeline = MLPipeline()
        self.command_history = []
    
    def execute_command(self, command):
        result = command.execute()
        self.command_history.append(command)
        return result
    
    def undo_last_command(self):
        if self.command_history:
            command = self.command_history.pop()
            command.undo()
        else:
            print("실행할 명령이 없습니다")

# 테스트: ML 실험 관리
print("=== ML 실험 관리 시스템 데모 ===")

manager = ExperimentManager()

# 1. 모델 학습
train_cmd = TrainModelCommand(manager.pipeline, "model_v1", "random_forest", {"n_estimators": 100})
model_info = manager.execute_command(train_cmd)
print(f"학습된 모델: {model_info}")

# 2. 모델 평가
eval_cmd = EvaluateModelCommand(manager.pipeline, "model_v1")
metrics = manager.execute_command(eval_cmd)
print(f"평가 결과: {metrics}")

# 3. 모델 배포
deploy_cmd = DeployModelCommand(manager.pipeline, "model_v1")
deployed = manager.execute_command(deploy_cmd)
print(f"배포된 모델: {deployed}")

print(f"\n현재 배포 상태: {manager.pipeline.deployed_model}")

# 4. 배포 취소 (undo)
print("\n=== 마지막 명령 취소 ===")
manager.undo_last_command()
print(f"취소 후 배포 상태: {manager.pipeline.deployed_model}")

print("\n🎯 Command 패턴으로 ML 작업을 객체화하여 undo/재실행 가능!")
print("   실험 재현성과 관리 용이")

=== ML 실험 관리 시스템 데모 ===
모델 학습 시작: model_v1 (random_forest)
✓ 모델 학습 완료: model_v1
학습된 모델: {'algorithm': 'random_forest', 'params': {'n_estimators': 100}, 'trained_at': 1769956147.3015034, 'status': 'trained'}
모델 평가 시작: model_v1
✓ 모델 평가 완료: model_v1
평가 결과: {'accuracy': 0.8874540118847363, 'precision': 0.9150714306409916, 'recall': 0.9531993941811405}
모델 배포 시작: model_v1
✓ 모델 배포 완료: model_v1
배포된 모델: model_v1

현재 배포 상태: model_v1

=== 마지막 명령 취소 ===
모델 배포 취소: model_v1
취소 후 배포 상태: None

🎯 Command 패턴으로 ML 작업을 객체화하여 undo/재실행 가능!
   실험 재현성과 관리 용이


### 3-3. Interpreter 패턴

Interpreter 패턴은 **"주어진 언어(문법)를 해석하는 방법을 클래스로 표현하여 문장을 해석(실행)하는 패턴"**입니다.

**비유로 이해하기:**
- **계산기**: "2 + 3 * 4"라는 문자열을 해석하여 계산
- 각 숫자와 연산자는 별도 클래스
- 트리 구조로 표현되어 계산됨

**코드로 이해하기:**
- `Expression`: 표현식 인터페이스 (interpret 메서드)
- `TerminalExpression`: 단말 표현식 (숫자, 변수)
- `NonTerminalExpression`: 비단말 표현식 (연산자)
- 문법 트리를 구성하여 해석

**머신러닝 엔지니어 관점:**
간단한 규칙 기반 시스템이나 설정 파일 파싱에 유용합니다. 예를 들어, "if feature1 > 0.5 and feature2 < 0.3 then predict positive" 같은 규칙을 해석할 수 있습니다.

**장점:**
- 문법을 클래스 구조로 표현해 확장 가능
- 새로운 규칙 쉽게 추가

**단점:**
- 복잡한 언어에는 구조가 무거움
- 문법 정의가 번거로움

In [15]:
# Interpreter 패턴 더 자세한 예제: 규칙 기반 분류기
# 머신러닝에서 Interpreter 패턴 적용 예시

from abc import ABC, abstractmethod

# Expression 인터페이스
class RuleExpression(ABC):
    @abstractmethod
    def interpret(self, context):
        """규칙을 해석하여 True/False 반환"""
        pass

# Terminal Expressions: 기본 조건들
class NumberExpression(RuleExpression):
    def __init__(self, value):
        self.value = float(value)
    
    def interpret(self, context):
        return self.value

class VariableExpression(RuleExpression):
    def __init__(self, variable_name):
        self.variable_name = variable_name
    
    def interpret(self, context):
        if self.variable_name not in context:
            raise ValueError(f"변수 '{self.variable_name}'가 context에 없습니다")
        return context[self.variable_name]

# Non-terminal Expressions: 연산자들
class BinaryOperation(RuleExpression):
    def __init__(self, left, right):
        self.left = left
        self.right = right

class GreaterThan(BinaryOperation):
    def interpret(self, context):
        return self.left.interpret(context) > self.right.interpret(context)

class LessThan(BinaryOperation):
    def interpret(self, context):
        return self.left.interpret(context) < self.right.interpret(context)

class Equal(BinaryOperation):
    def interpret(self, context):
        return self.left.interpret(context) == self.right.interpret(context)

class AndOperation(BinaryOperation):
    def interpret(self, context):
        return self.left.interpret(context) and self.right.interpret(context)

class OrOperation(BinaryOperation):
    def interpret(self, context):
        return self.left.interpret(context) or self.right.interpret(context)

# 규칙 엔진: 여러 규칙을 조합하여 분류
class RuleEngine:
    def __init__(self):
        self.rules = {}  # 규칙 이름 -> (조건, 결과)
    
    def add_rule(self, name, condition, result):
        self.rules[name] = (condition, result)
    
    def classify(self, context):
        """모든 규칙을 평가하여 매칭되는 결과 반환"""
        results = []
        for rule_name, (condition, result) in self.rules.items():
            try:
                if condition.interpret(context):
                    results.append((rule_name, result))
            except Exception as e:
                print(f"규칙 '{rule_name}' 평가 중 에러: {e}")
                continue
        return results

# 규칙 파서: 문자열 규칙을 Expression 트리로 변환
class RuleParser:
    def parse(self, rule_string):
        """간단한 규칙 파싱 (실제로는 더 복잡한 파서 필요)"""
        # 예: "age > 30 and income > 50000"
        tokens = rule_string.replace('(', ' ( ').replace(')', ' ) ').split()
        
        def parse_expression(index):
            token = tokens[index]
            
            if token.isdigit() or (token.replace('.', '').isdigit()):
                return NumberExpression(token), index + 1
            elif token in ['age', 'income', 'score']:  # 변수들
                return VariableExpression(token), index + 1
            elif token == '(':
                expr, next_index = parse_expression(index + 1)
                if tokens[next_index] != ')':
                    raise ValueError("괄호 불일치")
                return expr, next_index + 1
            elif token in ['>', '<', '==']:
                left, next_index = parse_expression(index - 1)  # 왼쪽으로 돌아감 (단순화)
                right, next_index = parse_expression(next_index)
                
                if token == '>':
                    return GreaterThan(left, right), next_index
                elif token == '<':
                    return LessThan(left, right), next_index
                elif token == '==':
                    return Equal(left, right), next_index
            elif token in ['and', 'or']:
                # 이진 연산자 처리 (단순화)
                left = None  # 이전 표현식
                right, next_index = parse_expression(index + 1)
                
                if token == 'and':
                    return AndOperation(left, right), next_index
                elif token == 'or':
                    return OrOperation(left, right), next_index
            
            raise ValueError(f"알 수 없는 토큰: {token}")
        
        # 간단한 파싱 (실제로는 토크나이저와 파서가 필요)
        # 여기서는 하드코딩된 규칙으로 대체
        if "age > 30 and income > 50000" in rule_string:
            return AndOperation(
                GreaterThan(VariableExpression("age"), NumberExpression("30")),
                GreaterThan(VariableExpression("income"), NumberExpression("50000"))
            )
        elif "score > 0.8" in rule_string:
            return GreaterThan(VariableExpression("score"), NumberExpression("0.8"))
        
        raise ValueError(f"지원하지 않는 규칙: {rule_string}")

# 테스트: 신용 승인 규칙 기반 분류기
print("=== 규칙 기반 분류기 데모 ===")

# 규칙 엔진 생성
engine = RuleEngine()
parser = RuleParser()

# 규칙 추가
rule1_condition = parser.parse("age > 30 and income > 50000")
engine.add_rule("고소득_성인", rule1_condition, "승인")

rule2_condition = parser.parse("score > 0.8")
engine.add_rule("고점수", rule2_condition, "승인")

# 테스트 데이터
test_cases = [
    {"age": 25, "income": 30000, "score": 0.6, "name": "청년_저소득"},
    {"age": 35, "income": 60000, "score": 0.9, "name": "성인_고소득_고점수"},
    {"age": 45, "income": 80000, "score": 0.7, "name": "중장년_고소득"},
    {"age": 28, "income": 40000, "score": 0.85, "name": "청년_고점수"}
]

for case in test_cases:
    context = {k: v for k, v in case.items() if k != 'name'}
    results = engine.classify(context)
    
    print(f"\n케이스: {case['name']}")
    print(f"데이터: {context}")
    print(f"매칭 규칙: {[rule for rule, _ in results]}")
    print(f"결과: {'승인' if results else '거부'}")

print("\n🎯 Interpreter 패턴으로 규칙 기반 ML 모델 구현!")
print("   복잡한 비즈니스 규칙을 유연하게 해석")

=== 규칙 기반 분류기 데모 ===

케이스: 청년_저소득
데이터: {'age': 25, 'income': 30000, 'score': 0.6}
매칭 규칙: []
결과: 거부

케이스: 성인_고소득_고점수
데이터: {'age': 35, 'income': 60000, 'score': 0.9}
매칭 규칙: ['고소득_성인', '고점수']
결과: 승인

케이스: 중장년_고소득
데이터: {'age': 45, 'income': 80000, 'score': 0.7}
매칭 규칙: ['고소득_성인']
결과: 승인

케이스: 청년_고점수
데이터: {'age': 28, 'income': 40000, 'score': 0.85}
매칭 규칙: ['고점수']
결과: 승인

🎯 Interpreter 패턴으로 규칙 기반 ML 모델 구현!
   복잡한 비즈니스 규칙을 유연하게 해석


### 3-4. Iterator 패턴

Iterator 패턴은 **"컬렉션 내부 구조를 드러내지 않고 순회하는 방법을 표준화하는 패턴"**입니다.

**비유로 이해하기:**
- **TV 채널**: 채널 업/다운 버튼으로 채널을 바꾸지만, TV의 내부 채널 목록은 모름
- 이터레이터가 다음 채널을 알려줌

**코드로 이해하기:**
- `Iterator`: 순회 인터페이스 (__next__, __iter__)
- `Iterable`: 이터레이터를 반환하는 객체
- `ConcreteIterator`: 구체적인 순회 로직

**머신러닝 엔지니어 관점:**
대용량 데이터셋을 메모리 효율적으로 순회할 때 유용합니다. 예를 들어, 수백GB의 데이터를 배치 단위로 읽어서 모델 학습에 사용.

**장점:**
- 순회 로직을 분리해 코드 단순화
- 다양한 컬렉션 타입을 동일한 방식으로 순회
- 메모리 효율적 순회 가능

**단점:**
- 파이썬은 기본 이터레이터 지원으로 불필요하게 복잡해질 수 있음

In [16]:
# Iterator 패턴 더 자세한 예제: 데이터 배치 이터레이터
# 머신러닝에서 Iterator 패턴 적용 예시

import pandas as pd
import numpy as np
from typing import Iterator, Iterable

class DataBatchIterator(Iterator):
    """데이터 배치를 순회하는 이터레이터"""
    def __init__(self, data_source, batch_size=32):
        self.data_source = data_source
        self.batch_size = batch_size
        self.current_index = 0
    
    def __next__(self):
        if self.current_index >= len(self.data_source):
            raise StopIteration
        
        # 배치 추출
        end_index = min(self.current_index + self.batch_size, len(self.data_source))
        batch = self.data_source.iloc[self.current_index:end_index]
        
        self.current_index = end_index
        return batch

class LargeDataset(Iterable):
    """대용량 데이터셋 (메모리에 전체 로드하지 않음)"""
    def __init__(self, file_path, chunk_size=1000):
        self.file_path = file_path
        self.chunk_size = chunk_size
        self._data_cache = None
        self._current_chunk = 0
    
    def __iter__(self):
        """새로운 이터레이터 반환"""
        return DataBatchIterator(self._load_data(), batch_size=32)
    
    def _load_data(self):
        """데이터를 청크 단위로 로드 (메모리 효율적)"""
        if self._data_cache is None:
            print(f"데이터 파일 로드: {self.file_path}")
            # 실제로는 큰 파일을 청크로 읽음
            # 여기서는 샘플 데이터 생성
            np.random.seed(42)
            self._data_cache = pd.DataFrame({
                'feature1': np.random.randn(1000),
                'feature2': np.random.randn(1000),
                'feature3': np.random.randn(1000),
                'target': np.random.randint(0, 2, 1000)
            })
        return self._data_cache

class StreamingDataset(Iterable):
    """스트리밍 데이터셋: 파일에서 직접 배치 읽기"""
    def __init__(self, file_path, batch_size=32):
        self.file_path = file_path
        self.batch_size = batch_size
    
    def __iter__(self):
        return StreamingIterator(self.file_path, self.batch_size)

class StreamingIterator(Iterator):
    """파일에서 직접 배치를 스트리밍하는 이터레이터"""
    def __init__(self, file_path, batch_size):
        self.file_path = file_path
        self.batch_size = batch_size
        self.chunk_iter = pd.read_csv(file_path, chunksize=batch_size)
        self.current_chunk = None
    
    def __next__(self):
        try:
            self.current_chunk = next(self.chunk_iter)
            return self.current_chunk
        except StopIteration:
            raise StopIteration

# 모델 학습 시뮬레이션 함수
def train_on_batch(batch, epoch, batch_idx):
    """배치 데이터로 모델 학습 (시뮬레이션)"""
    X = batch[['feature1', 'feature2', 'feature3']].values
    y = batch['target'].values
    
    # 간단한 학습 시뮬레이션
    loss = np.random.random() * 0.5 + 0.1
    accuracy = np.random.random() * 0.3 + 0.7
    
    print(f"Epoch {epoch}, Batch {batch_idx}: Loss={loss:.3f}, Accuracy={accuracy:.3f}")
    return loss, accuracy

# 테스트: 데이터 배치 순회
print("=== 데이터 배치 이터레이터 데모 ===")

# 1. 메모리에 로드된 데이터셋
dataset = LargeDataset("large_dataset.csv")

print("메모리 로드 데이터셋 순회:")
batch_losses = []
for epoch in range(2):  # 2 에폭
    batch_idx = 0
    for batch in dataset:  # 각 에폭마다 새로운 이터레이터
        loss, acc = train_on_batch(batch, epoch + 1, batch_idx)
        batch_losses.append(loss)
        batch_idx += 1
        if batch_idx >= 5:  # 처음 5개 배치만
            break

print(f"\n평균 손실: {np.mean(batch_losses):.3f}")

# 2. 스트리밍 데이터셋 (파일에서 직접)
print("\n=== 스트리밍 데이터셋 데모 ===")
# 실제 파일이 없으므로 샘플 데이터로 대체
streaming_data = pd.DataFrame({
    'feature1': np.random.randn(100),
    'feature2': np.random.randn(100),
    'feature3': np.random.randn(100),
    'target': np.random.randint(0, 2, 100)
})

# CSV로 저장 후 스트리밍
temp_file = "temp_dataset.csv"
streaming_data.to_csv(temp_file, index=False)

streaming_dataset = StreamingDataset(temp_file, batch_size=10)

print("스트리밍 데이터셋 순회:")
total_samples = 0
for batch in streaming_dataset:
    print(f"배치 크기: {len(batch)}, 누적 샘플: {total_samples + len(batch)}")
    total_samples += len(batch)
    if total_samples >= 50:  # 50개 샘플만
        break

# 임시 파일 정리
import os
os.remove(temp_file)

print("\n🎯 Iterator 패턴으로 대용량 데이터 메모리 효율적 순회!")
print("   배치 학습에 적합한 데이터 파이프라인 구축")

=== 데이터 배치 이터레이터 데모 ===
메모리 로드 데이터셋 순회:
데이터 파일 로드: large_dataset.csv
Epoch 1, Batch 0: Loss=0.478, Accuracy=0.876
Epoch 1, Batch 1: Loss=0.514, Accuracy=0.724
Epoch 1, Batch 2: Loss=0.338, Accuracy=0.889
Epoch 1, Batch 3: Loss=0.514, Accuracy=0.935
Epoch 1, Batch 4: Loss=0.239, Accuracy=0.982
Epoch 2, Batch 0: Loss=0.162, Accuracy=0.963
Epoch 2, Batch 1: Loss=0.585, Accuracy=0.753
Epoch 2, Batch 2: Loss=0.461, Accuracy=0.712
Epoch 2, Batch 3: Loss=0.303, Accuracy=0.855
Epoch 2, Batch 4: Loss=0.390, Accuracy=0.981

평균 손실: 0.398

=== 스트리밍 데이터셋 데모 ===
스트리밍 데이터셋 순회:
배치 크기: 10, 누적 샘플: 10
배치 크기: 10, 누적 샘플: 20
배치 크기: 10, 누적 샘플: 30
배치 크기: 10, 누적 샘플: 40
배치 크기: 10, 누적 샘플: 50

🎯 Iterator 패턴으로 대용량 데이터 메모리 효율적 순회!
   배치 학습에 적합한 데이터 파이프라인 구축


### 3-5. Mediator 패턴

Mediator 패턴은 **"객체들이 직접 서로 참조하며 통신하는 것을 피하기 위해 중재자를 두고 통신을 중앙에서 관리하는 패턴"**입니다.

**비유로 이해하기:**
- **항공 교통 관제**: 비행기들이 서로 직접 통신하지 않고 관제탑을 통해 조율
- 각 비행기는 관제탑만 알면 되고, 다른 비행기의 위치는 관제탑이 관리

**코드로 이해하기:**
- `Mediator`: 중재자 인터페이스
- `Colleague`: 동료 객체들 (서로 직접 통신하지 않음)
- `ConcreteMediator`: 구체적인 중재 로직

**머신러닝 엔지니어 관점:**
복잡한 ML 시스템에서 컴포넌트 간 결합도를 낮출 때 유용합니다. 예를 들어, 데이터 로더, 모델, 평가자가 서로 직접 통신하지 않고 mediator를 통해 협력.

**장점:**
- 객체 간 결합도 감소
- 중앙 집중식 통신 관리
- 새로운 컴포넌트 쉽게 추가

**단점:**
- mediator가 비대해질 수 있음
- 단일 실패 지점 발생

In [17]:
# Mediator 패턴 더 자세한 예제: 모델 앙상블 시스템
# 머신러닝에서 Mediator 패턴 적용 예시

from abc import ABC, abstractmethod
import numpy as np

# Mediator 인터페이스
class EnsembleMediator(ABC):
    @abstractmethod
    def register_model(self, model):
        pass
    
    @abstractmethod
    def send_prediction(self, model, prediction, confidence):
        pass
    
    @abstractmethod
    def get_final_prediction(self, input_data):
        pass

# Colleague: 개별 모델들
class BaseModel(ABC):
    def __init__(self, mediator, name):
        self.mediator = mediator
        self.name = name
        self.mediator.register_model(self)
    
    @abstractmethod
    def predict(self, input_data):
        pass

class RandomForestModel(BaseModel):
    def __init__(self, mediator):
        super().__init__(mediator, "RandomForest")
        self.accuracy = 0.85  # 가상의 정확도
    
    def predict(self, input_data):
        # RF 예측 시뮬레이션
        prediction = np.random.choice([0, 1], p=[0.4, 0.6])
        confidence = self.accuracy + np.random.random() * 0.1 - 0.05
        
        # mediator를 통해 예측 전송
        self.mediator.send_prediction(self, prediction, confidence)
        return prediction, confidence

class SVMModel(BaseModel):
    def __init__(self, mediator):
        super().__init__(mediator, "SVM")
        self.accuracy = 0.82
    
    def predict(self, input_data):
        # SVM 예측 시뮬레이션
        prediction = np.random.choice([0, 1], p=[0.3, 0.7])
        confidence = self.accuracy + np.random.random() * 0.1 - 0.05
        
        self.mediator.send_prediction(self, prediction, confidence)
        return prediction, confidence

class NeuralNetworkModel(BaseModel):
    def __init__(self, mediator):
        super().__init__(mediator, "NeuralNetwork")
        self.accuracy = 0.88
    
    def predict(self, input_data):
        # NN 예측 시뮬레이션
        prediction = np.random.choice([0, 1], p=[0.2, 0.8])
        confidence = self.accuracy + np.random.random() * 0.1 - 0.05
        
        self.mediator.send_prediction(self, prediction, confidence)
        return prediction, confidence

# Concrete Mediator: 앙상블 전략 관리
class VotingEnsembleMediator(EnsembleMediator):
    def __init__(self, strategy="weighted"):
        self.models = []
        self.strategy = strategy  # "majority" 또는 "weighted"
        self.current_predictions = {}  # 현재 예측들을 저장
    
    def register_model(self, model):
        self.models.append(model)
        print(f"모델 등록: {model.name}")
    
    def send_prediction(self, model, prediction, confidence):
        self.current_predictions[model.name] = {
            'prediction': prediction,
            'confidence': confidence
        }
        print(f"{model.name} 예측 수신: {prediction} (신뢰도: {confidence:.2f})")
    
    def get_final_prediction(self, input_data):
        if not self.models:
            raise ValueError("등록된 모델이 없습니다")
        
        # 각 모델에 예측 요청
        self.current_predictions = {}
        for model in self.models:
            model.predict(input_data)
        
        # 앙상블 전략에 따라 최종 예측
        if self.strategy == "majority":
            return self._majority_vote()
        elif self.strategy == "weighted":
            return self._weighted_vote()
        else:
            raise ValueError(f"지원하지 않는 전략: {self.strategy}")
    
    def _majority_vote(self):
        """단순 다수결"""
        predictions = [pred['prediction'] for pred in self.current_predictions.values()]
        final_pred = 1 if sum(predictions) > len(predictions) / 2 else 0
        return final_pred, "majority_vote"
    
    def _weighted_vote(self):
        """신뢰도로 가중치 적용"""
        total_weight = 0
        weighted_sum = 0
        
        for pred_info in self.current_predictions.values():
            weight = pred_info['confidence']
            weighted_sum += pred_info['prediction'] * weight
            total_weight += weight
        
        final_pred = 1 if weighted_sum / total_weight > 0.5 else 0
        return final_pred, "weighted_vote"

# 테스트: 앙상블 시스템
print("=== 모델 앙상블 시스템 데모 ===")

# Mediator 생성
mediator = VotingEnsembleMediator(strategy="weighted")

# 모델들 등록 (서로 직접 알지 못함)
rf_model = RandomForestModel(mediator)
svm_model = SVMModel(mediator)
nn_model = NeuralNetworkModel(mediator)

print(f"\n등록된 모델들: {[model.name for model in mediator.models]}")

# 테스트 데이터로 앙상블 예측
test_inputs = [
    [0.1, 0.2, 0.3],  # 테스트 케이스 1
    [0.8, 0.9, 0.7],  # 테스트 케이스 2
]

for i, input_data in enumerate(test_inputs, 1):
    print(f"\n--- 테스트 케이스 {i} ---")
    final_pred, strategy = mediator.get_final_prediction(input_data)
    print(f"최종 앙상블 예측: {final_pred} (전략: {strategy})")
    
    # 개별 모델 예측 결과 출력
    for model_name, pred_info in mediator.current_predictions.items():
        print(f"  {model_name}: {pred_info['prediction']} (신뢰도: {pred_info['confidence']:.2f})")

print("\n🎯 Mediator 패턴으로 모델 간 결합도 제거!")
print("   새로운 모델 쉽게 추가하고 앙상블 전략 변경 가능")

=== 모델 앙상블 시스템 데모 ===
모델 등록: RandomForest
모델 등록: SVM
모델 등록: NeuralNetwork

등록된 모델들: ['RandomForest', 'SVM', 'NeuralNetwork']

--- 테스트 케이스 1 ---
RandomForest 예측 수신: 1 (신뢰도: 0.83)
SVM 예측 수신: 1 (신뢰도: 0.83)
NeuralNetwork 예측 수신: 1 (신뢰도: 0.84)
최종 앙상블 예측: 1 (전략: weighted_vote)
  RandomForest: 1 (신뢰도: 0.83)
  SVM: 1 (신뢰도: 0.83)
  NeuralNetwork: 1 (신뢰도: 0.84)

--- 테스트 케이스 2 ---
RandomForest 예측 수신: 0 (신뢰도: 0.90)
SVM 예측 수신: 0 (신뢰도: 0.78)
NeuralNetwork 예측 수신: 0 (신뢰도: 0.83)
최종 앙상블 예측: 0 (전략: weighted_vote)
  RandomForest: 0 (신뢰도: 0.90)
  SVM: 0 (신뢰도: 0.78)
  NeuralNetwork: 0 (신뢰도: 0.83)

🎯 Mediator 패턴으로 모델 간 결합도 제거!
   새로운 모델 쉽게 추가하고 앙상블 전략 변경 가능


### 3-6. Memento 패턴

Memento 패턴은 **"객체의 내부 상태를 외부에 저장해 두고(스냅샷), 나중에 그 상태로 되돌리는 패턴"**입니다.

**비유로 이해하기:**
- **게임 세이브**: 게임 진행 상황을 저장했다가 나중에 불러옴
- 게임 캐릭터의 위치, 체력, 아이템 상태를 캡슐화하여 저장

**코드로 이해하기:**
- `Originator`: 상태를 저장/복원하는 객체
- `Memento`: 상태 스냅샷 (불변 객체)
- `Caretaker`: Memento를 관리하는 객체

**머신러닝 엔지니어 관점:**
모델 학습 중간 상태를 저장하여 재개하거나, 실험 설정을 백업하여 재현할 때 유용합니다. 예를 들어, 긴 학습 과정에서 체크포인트를 저장.

**장점:**
- 캡슐화 유지하면서 상태 복원 가능
- originator의 내부 구조 노출하지 않음

**단점:**
- 상태 저장 비용 (메모리/디스크)
- 많은 memento 관리 복잡

In [18]:
# Memento 패턴 더 자세한 예제: 모델 학습 체크포인트
# 머신러닝에서 Memento 패턴 적용 예시

import copy
import time
from typing import Dict, Any

# Memento: 모델 상태 스냅샷
class ModelMemento:
    def __init__(self, state: Dict[str, Any]):
        self._state = copy.deepcopy(state)  # 깊은 복사로 불변성 보장
        self._timestamp = time.time()
    
    @property
    def state(self) -> Dict[str, Any]:
        return copy.deepcopy(self._state)  # 복사본 반환으로 불변성 유지
    
    @property
    def timestamp(self) -> float:
        return self._timestamp

# Originator: 모델 학습 상태 관리
class MLModel:
    def __init__(self, name: str):
        self.name = name
        self.weights = {"layer1": [0.1, 0.2, 0.3], "layer2": [0.4, 0.5]}
        self.learning_rate = 0.01
        self.epoch = 0
        self.loss_history = []
        self.best_loss = float('inf')
    
    def save_checkpoint(self) -> ModelMemento:
        """현재 상태를 Memento로 저장"""
        state = {
            'weights': self.weights,
            'learning_rate': self.learning_rate,
            'epoch': self.epoch,
            'loss_history': self.loss_history,
            'best_loss': self.best_loss
        }
        print(f"체크포인트 저장: 에폭 {self.epoch}, 손실 {self.best_loss:.4f}")
        return ModelMemento(state)
    
    def restore_checkpoint(self, memento: ModelMemento):
        """Memento에서 상태 복원"""
        state = memento.state
        self.weights = state['weights']
        self.learning_rate = state['learning_rate']
        self.epoch = state['epoch']
        self.loss_history = state['loss_history']
        self.best_loss = state['best_loss']
        print(f"체크포인트 복원: 에폭 {self.epoch}, 손실 {self.best_loss:.4f}")
    
    def train_step(self):
        """학습 스텝 시뮬레이션"""
        self.epoch += 1
        
        # 가상의 손실 계산
        current_loss = 1.0 / (self.epoch + 1) + np.random.random() * 0.1
        self.loss_history.append(current_loss)
        
        # 최고 성능 업데이트
        if current_loss < self.best_loss:
            self.best_loss = current_loss
            print(f"✓ 최고 성능 업데이트: {current_loss:.4f}")
        
        # 가중치 업데이트 시뮬레이션
        for layer in self.weights:
            self.weights[layer] = [w + np.random.random() * 0.01 - 0.005 for w in self.weights[layer]]
        
        return current_loss
    
    def get_status(self):
        return {
            'epoch': self.epoch,
            'current_loss': self.loss_history[-1] if self.loss_history else None,
            'best_loss': self.best_loss,
            'learning_rate': self.learning_rate
        }

# Caretaker: 체크포인트 관리
class CheckpointManager:
    def __init__(self):
        self._checkpoints: Dict[str, ModelMemento] = {}
    
    def save_checkpoint(self, name: str, memento: ModelMemento):
        self._checkpoints[name] = memento
        print(f"체크포인트 '{name}' 저장됨")
    
    def load_checkpoint(self, name: str) -> ModelMemento:
        if name not in self._checkpoints:
            raise ValueError(f"체크포인트 '{name}'이 존재하지 않습니다")
        return self._checkpoints[name]
    
    def list_checkpoints(self):
        return list(self._checkpoints.keys())
    
    def get_checkpoint_info(self, name: str):
        if name not in self._checkpoints:
            return None
        memento = self._checkpoints[name]
        state = memento.state
        return {
            'name': name,
            'epoch': state['epoch'],
            'best_loss': state['best_loss'],
            'timestamp': memento.timestamp
        }

# 테스트: 모델 학습 체크포인트 시스템
print("=== 모델 학습 체크포인트 데모 ===")

# 모델과 체크포인트 관리자 생성
model = MLModel("sentiment_classifier")
checkpoint_manager = CheckpointManager()

# 학습 시뮬레이션
print("학습 시작...")
for i in range(10):
    loss = model.train_step()
    print(f"에폭 {model.epoch}: 손실 = {loss:.4f}")
    
    # 특정 에폭마다 체크포인트 저장
    if model.epoch in [3, 6, 9]:
        checkpoint = model.save_checkpoint()
        checkpoint_manager.save_checkpoint(f"epoch_{model.epoch}", checkpoint)

print(f"\n최종 상태: {model.get_status()}")

# 체크포인트 목록 확인
print(f"\n저장된 체크포인트: {checkpoint_manager.list_checkpoints()}")

# 체크포인트 정보 출력
for cp_name in checkpoint_manager.list_checkpoints():
    info = checkpoint_manager.get_checkpoint_info(cp_name)
    print(f"체크포인트 '{cp_name}': 에폭 {info['epoch']}, 최고 손실 {info['best_loss']:.4f}")

# 특정 체크포인트로 복원
print("\n=== 체크포인트 복원 테스트 ===")
restore_checkpoint = checkpoint_manager.load_checkpoint("epoch_6")
model.restore_checkpoint(restore_checkpoint)

print(f"복원 후 상태: {model.get_status()}")

# 추가 학습 이어서 진행
print("\n추가 학습...")
for i in range(3):
    loss = model.train_step()
    print(f"에폭 {model.epoch}: 손실 = {loss:.4f}")

print(f"\n최종 상태: {model.get_status()}")

print("\n🎯 Memento 패턴으로 모델 학습 상태 저장/복원!")
print("   긴 학습 과정에서 중단/재개 및 실험 재현 가능")

=== 모델 학습 체크포인트 데모 ===
학습 시작...
✓ 최고 성능 업데이트: 0.5392
에폭 1: 손실 = 0.5392
✓ 최고 성능 업데이트: 0.4195
에폭 2: 손실 = 0.4195
✓ 최고 성능 업데이트: 0.2983
에폭 3: 손실 = 0.2983
체크포인트 저장: 에폭 3, 손실 0.2983
체크포인트 'epoch_3' 저장됨
✓ 최고 성능 업데이트: 0.2858
에폭 4: 손실 = 0.2858
✓ 최고 성능 업데이트: 0.1901
에폭 5: 손실 = 0.1901
✓ 최고 성능 업데이트: 0.1670
에폭 6: 손실 = 0.1670
체크포인트 저장: 에폭 6, 손실 0.1670
체크포인트 'epoch_6' 저장됨
✓ 최고 성능 업데이트: 0.1667
에폭 7: 손실 = 0.1667
✓ 최고 성능 업데이트: 0.1218
에폭 8: 손실 = 0.1218
✓ 최고 성능 업데이트: 0.1164
에폭 9: 손실 = 0.1164
체크포인트 저장: 에폭 9, 손실 0.1164
체크포인트 'epoch_9' 저장됨
에폭 10: 손실 = 0.1819

최종 상태: {'epoch': 10, 'current_loss': 0.18185204127767857, 'best_loss': 0.11642868432724093, 'learning_rate': 0.01}

저장된 체크포인트: ['epoch_3', 'epoch_6', 'epoch_9']
체크포인트 'epoch_3': 에폭 3, 최고 손실 0.2983
체크포인트 'epoch_6': 에폭 6, 최고 손실 0.1670
체크포인트 'epoch_9': 에폭 9, 최고 손실 0.1164

=== 체크포인트 복원 테스트 ===
체크포인트 복원: 에폭 6, 손실 0.1670
복원 후 상태: {'epoch': 6, 'current_loss': 0.16697366428791904, 'best_loss': 0.16697366428791904, 'learning_rate': 0.01}

추가 학습...
✓ 최고 성능 업데이트: 0.

### 3-7. Observer (Publish–Subscribe) 패턴

Observer 패턴은 **"한 객체(subject)의 상태 변화를 여러 관찰자(observer)에게 자동으로 알리는 패턴"**입니다.

**비유로 이해하기:**
- **신문 구독**: 신문사가 새 기사를 출판하면 모든 구독자에게 자동 배송
- 신문사는 구독자 목록만 관리하고, 각 구독자는 독립적으로 기사 처리

**코드로 이해하기:**
- `Subject`: 관찰 대상 (상태 변경을 알림)
- `Observer`: 관찰자 인터페이스
- `ConcreteObserver`: 구체적인 관찰자들

**머신러닝 엔지니어 관점:**
모델 학습 진행 상황을 실시간으로 모니터링할 때 유용합니다. 예를 들어, 학습 중 손실 감소, 정확도 상승 등을 여러 모니터링 도구에 자동 알림.

**장점:**
- 느슨한 결합 (subject와 observer가 서로 모름)
- 다수의 observer 동적 추가/제거 가능
- 일대다 의존성 관리

**단점:**
- 예상치 못한 업데이트 순서
- 디버깅 복잡성 (observer가 많을 때)

In [19]:
# Observer 패턴 더 자세한 예제: 모델 학습 모니터링 시스템
# 머신러닝에서 Observer 패턴 적용 예시

from abc import ABC, abstractmethod
from typing import Dict, Any
import time

# Observer 인터페이스
class TrainingObserver(ABC):
    @abstractmethod
    def update(self, metrics: Dict[str, Any]):
        pass

# Subject: 모델 학습 트레이너
class ModelTrainer:
    def __init__(self):
        self._observers = []
        self.current_metrics = {
            'epoch': 0,
            'loss': 0.0,
            'accuracy': 0.0,
            'val_loss': 0.0,
            'val_accuracy': 0.0
        }
    
    def attach(self, observer: TrainingObserver):
        """옵저버 등록"""
        self._observers.append(observer)
        print(f"옵저버 등록: {observer.__class__.__name__}")
    
    def detach(self, observer: TrainingObserver):
        """옵저버 제거"""
        self._observers.remove(observer)
        print(f"옵저버 제거: {observer.__class__.__name__}")
    
    def notify(self):
        """모든 옵저버에게 알림"""
        for observer in self._observers:
            observer.update(self.current_metrics.copy())
    
    def update_metrics(self, metrics: Dict[str, Any]):
        """메트릭 업데이트 및 알림"""
        self.current_metrics.update(metrics)
        self.notify()
    
    def train_epoch(self, epoch: int):
        """학습 에폭 시뮬레이션"""
        # 가상의 메트릭 생성
        loss = 1.0 / (epoch + 1) + np.random.random() * 0.1
        accuracy = min(0.95, 0.5 + epoch * 0.1 + np.random.random() * 0.05)
        val_loss = loss + np.random.random() * 0.2 - 0.1
        val_accuracy = accuracy - np.random.random() * 0.1
        
        metrics = {
            'epoch': epoch,
            'loss': loss,
            'accuracy': accuracy,
            'val_loss': val_loss,
            'val_accuracy': val_accuracy,
            'timestamp': time.time()
        }
        
        self.update_metrics(metrics)
        return metrics

# Concrete Observers
class LoggerObserver(TrainingObserver):
    """학습 로그 기록 옵저버"""
    def __init__(self, log_file="training.log"):
        self.log_file = log_file
        self.logs = []
    
    def update(self, metrics):
        log_entry = f"Epoch {metrics['epoch']}: loss={metrics['loss']:.4f}, acc={metrics['accuracy']:.4f}, val_loss={metrics['val_loss']:.4f}, val_acc={metrics['val_accuracy']:.4f}"
        self.logs.append(log_entry)
        print(f"[LOG] {log_entry}")
        
        # 파일에 저장 시뮬레이션
        with open(self.log_file, 'a') as f:
            f.write(log_entry + '\n')

class AlertObserver(TrainingObserver):
    """성능 알림 옵저버"""
    def __init__(self, alert_threshold=0.1):
        self.alert_threshold = alert_threshold
        self.previous_loss = float('inf')
    
    def update(self, metrics):
        current_loss = metrics['loss']
        
        # 손실이 증가하면 경고
        if current_loss > self.previous_loss + self.alert_threshold:
            print(f"⚠️ [ALERT] 손실 증가 감지! 이전: {self.previous_loss:.4f}, 현재: {current_loss:.4f}")
        
        # 정확도가 임계값 이상이면 축하
        if metrics['accuracy'] > 0.9:
            print(f"🎉 [ALERT] 높은 정확도 달성! {metrics['accuracy']:.4f}")
        
        self.previous_loss = current_loss

class VisualizationObserver(TrainingObserver):
    """학습 곡선 시각화 옵저버"""
    def __init__(self):
        self.history = {
            'epochs': [],
            'loss': [],
            'accuracy': [],
            'val_loss': [],
            'val_accuracy': []
        }
    
    def update(self, metrics):
        # 히스토리 저장
        for key in self.history:
            if key in metrics:
                self.history[key].append(metrics[key])
        
        # 간단한 텍스트 기반 시각화
        if len(self.history['epochs']) > 1:
            print(f"[VIS] 학습 곡선 업데이트 - 총 {len(self.history['epochs'])} 에폭")
            print(f"      최근 손실 추이: {self.history['loss'][-3:]}")

class SlackObserver(TrainingObserver):
    """Slack 알림 옵저버"""
    def __init__(self, webhook_url=None):
        self.webhook_url = webhook_url
    
    def update(self, metrics):
        if metrics['epoch'] % 5 == 0:  # 5 에폭마다 알림
            message = f"학습 진행 상황 - 에폭 {metrics['epoch']}\n정확도: {metrics['accuracy']:.2%}\n검증 정확도: {metrics['val_accuracy']:.2%}"
            print(f"[SLACK] {message}")
            
            # 실제로는 webhook으로 전송
            # requests.post(self.webhook_url, json={"text": message})

# 테스트: 학습 모니터링 시스템
print("=== 모델 학습 모니터링 시스템 데모 ===")

# 트레이너 생성
trainer = ModelTrainer()

# 다양한 옵저버 등록
logger = LoggerObserver()
alert = AlertObserver()
visualizer = VisualizationObserver()
slack = SlackObserver()

trainer.attach(logger)
trainer.attach(alert)
trainer.attach(visualizer)
trainer.attach(slack)

print(f"\n등록된 옵저버: {[obs.__class__.__name__ for obs in trainer._observers]}")

# 학습 시뮬레이션
print("\n=== 학습 시작 ===")
for epoch in range(1, 11):
    metrics = trainer.train_epoch(epoch)
    
    # 특정 시점에 옵저버 제거/추가
    if epoch == 7:
        print("\n--- 에폭 7: Slack 옵저버 제거 ---")
        trainer.detach(slack)
    
    time.sleep(0.1)  # 시뮬레이션 딜레이

print(f"\n최종 메트릭: {trainer.current_metrics}")

# 로그 확인
print(f"\n=== 학습 로그 (최근 3개) ===")
for log in logger.logs[-3:]:
    print(log)

print("\n🎯 Observer 패턴으로 실시간 학습 모니터링!")
print("   로깅, 알림, 시각화 옵저버 독립적으로 추가/제거 가능")

=== 모델 학습 모니터링 시스템 데모 ===
옵저버 등록: LoggerObserver
옵저버 등록: AlertObserver
옵저버 등록: VisualizationObserver
옵저버 등록: SlackObserver

등록된 옵저버: ['LoggerObserver', 'AlertObserver', 'VisualizationObserver', 'SlackObserver']

=== 학습 시작 ===
[LOG] Epoch 1: loss=0.5691, acc=0.6084, val_loss=0.5312, val_acc=0.5580
[LOG] Epoch 2: loss=0.4130, acc=0.7370, val_loss=0.4421, val_acc=0.7214
[LOG] Epoch 3: loss=0.3047, acc=0.8044, val_loss=0.2802, val_acc=0.7630
[LOG] Epoch 4: loss=0.2040, acc=0.9137, val_loss=0.2951, val_acc=0.8183
🎉 [ALERT] 높은 정확도 달성! 0.9137
[LOG] Epoch 5: loss=0.2019, acc=0.9500, val_loss=0.1377, val_acc=0.9108
🎉 [ALERT] 높은 정확도 달성! 0.9500
[SLACK] 학습 진행 상황 - 에폭 5
정확도: 95.00%
검증 정확도: 91.08%
[LOG] Epoch 6: loss=0.2376, acc=0.9500, val_loss=0.2903, val_acc=0.9391
🎉 [ALERT] 높은 정확도 달성! 0.9500
[LOG] Epoch 7: loss=0.2231, acc=0.9500, val_loss=0.2782, val_acc=0.9157
🎉 [ALERT] 높은 정확도 달성! 0.9500

--- 에폭 7: Slack 옵저버 제거 ---
옵저버 제거: SlackObserver
[LOG] Epoch 8: loss=0.1739, acc=0.9500, val_loss=0.1031, 

### 3-8. State 패턴

State 패턴은 **"객체의 내부 상태를 별도 객체로 캡슐화하여 상태에 따른 행동을 동적으로 변경하는 패턴"**입니다.

**비유로 이해하기:**
- **자동차 기어**: 주행 중 기어를 바꾸면 같은 액셀 페달을 밟아도 다른 반응
- D(주행), R(후진), P(주차) 상태에 따라 행동이 달라짐

**코드로 이해하기:**
- `Context`: 상태를 가지고 행동을 위임
- `State`: 상태 인터페이스
- `ConcreteState`: 구체적인 상태들

**머신러닝 엔지니어 관점:**
모델의 라이프사이클을 관리할 때 유용합니다. 예를 들어, 모델이 "학습 준비", "학습 중", "학습 완료", "추론 모드" 상태에 따라 다른 API를 제공.

**장점:**
- 상태별 로직을 분리하여 코드 가독성 향상
- 새로운 상태 쉽게 추가
- 상태 전환 로직을 한 곳에서 관리

**단점:**
- 상태가 많아지면 클래스 수가 증가
- 간단한 상태 변화에 오버엔지니어링

In [20]:
# State 패턴 더 자세한 예제: 모델 라이프사이클 관리
# 머신러닝에서 State 패턴 적용 예시

from abc import ABC, abstractmethod
import numpy as np

# State 인터페이스
class ModelState(ABC):
    @abstractmethod
    def train(self, context, data):
        pass
    
    @abstractmethod
    def predict(self, context, input_data):
        pass
    
    @abstractmethod
    def save(self, context, path):
        pass
    
    @abstractmethod
    def load(self, context, path):
        pass

# Concrete States
class InitializedState(ModelState):
    """모델 초기화 상태"""
    def train(self, context, data):
        print("모델 학습 시작...")
        # 학습 로직 시뮬레이션
        context.weights = np.random.randn(10, 1) * 0.1
        context.bias = 0.0
        context.is_trained = False
        context.change_state(TrainingState())
        print("✓ 학습 상태로 전환")
    
    def predict(self, context, input_data):
        raise RuntimeError("모델이 학습되지 않았습니다. 먼저 train()을 호출하세요.")
    
    def save(self, context, path):
        raise RuntimeError("저장할 모델이 없습니다.")
    
    def load(self, context, path):
        print(f"모델 로드: {path}")
        # 로드 시뮬레이션
        context.weights = np.random.randn(10, 1)
        context.bias = np.random.randn()
        context.is_trained = True
        context.change_state(TrainedState())
        print("✓ 학습 완료 상태로 전환")

class TrainingState(ModelState):
    """학습 중 상태"""
    def __init__(self):
        self.epochs_completed = 0
    
    def train(self, context, data):
        if self.epochs_completed < 5:  # 5 에폭 학습 시뮬레이션
            self.epochs_completed += 1
            print(f"학습 진행 중... 에폭 {self.epochs_completed}/5")
            
            # 가중치 업데이트 시뮬레이션
            context.weights += np.random.randn(*context.weights.shape) * 0.01
            context.bias += np.random.randn() * 0.01
            
            if self.epochs_completed >= 5:
                context.is_trained = True
                context.change_state(TrainedState())
                print("✓ 학습 완료! 추론 준비 상태로 전환")
        else:
            print("학습이 이미 완료되었습니다.")
    
    def predict(self, context, input_data):
        print("⚠️ 학습 중에는 예측이 제한됩니다. 학습 완료 후 사용하세요.")
        return None
    
    def save(self, context, path):
        print(f"체크포인트 저장: {path} (에폭 {self.epochs_completed})")
        # 실제로는 파일에 저장
    
    def load(self, context, path):
        raise RuntimeError("학습 중에는 로드할 수 없습니다.")

class TrainedState(ModelState):
    """학습 완료 상태"""
    def train(self, context, data):
        print("모델이 이미 학습되었습니다. 추가 학습을 원하시면 fine_tune()을 사용하세요.")
    
    def predict(self, context, input_data):
        # 선형 회귀 예측 시뮬레이션
        prediction = np.dot(input_data, context.weights) + context.bias
        print(f"예측 결과: {prediction.flatten()}")
        return prediction
    
    def save(self, context, path):
        print(f"학습된 모델 저장: {path}")
        # 실제로는 파일에 저장
    
    def load(self, context, path):
        raise RuntimeError("이미 학습된 모델입니다. 새 모델을 생성하세요.")

class InferenceState(ModelState):
    """추론 전용 상태 (최적화된 모드)"""
    def train(self, context, data):
        raise RuntimeError("추론 모드에서는 학습할 수 없습니다.")
    
    def predict(self, context, input_data):
        # 최적화된 추론 (배치 처리 등)
        prediction = np.dot(input_data, context.weights) + context.bias
        print(f"[추론 모드] 예측 결과: {prediction.flatten()}")
        return prediction
    
    def save(self, context, path):
        raise RuntimeError("추론 모드에서는 저장할 수 없습니다.")
    
    def load(self, context, path):
        raise RuntimeError("추론 모드에서는 로드할 수 없습니다.")

# Context: 모델
class MLModel:
    def __init__(self):
        self.weights = None
        self.bias = None
        self.is_trained = False
        self._state = InitializedState()
    
    def change_state(self, new_state: ModelState):
        self._state = new_state
        print(f"모델 상태 변경: {new_state.__class__.__name__}")
    
    def train(self, data):
        self._state.train(self, data)
    
    def predict(self, input_data):
        return self._state.predict(self, input_data)
    
    def save(self, path):
        self._state.save(self, path)
    
    def load(self, path):
        self._state.load(self, path)
    
    def set_inference_mode(self):
        if self.is_trained:
            self.change_state(InferenceState())
            print("✓ 추론 모드로 전환")
        else:
            raise RuntimeError("학습되지 않은 모델은 추론 모드로 전환할 수 없습니다.")

# 테스트: 모델 상태 관리
print("=== 모델 라이프사이클 관리 데모 ===")

model = MLModel()

# 1. 초기 상태에서 예측 시도 (실패)
print("1. 초기 상태에서 예측 시도:")
try:
    model.predict(np.array([[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]]))
except RuntimeError as e:
    print(f"에러: {e}")

# 2. 학습 시작
print("\n2. 학습 시작:")
train_data = np.random.randn(100, 10)  # 가상의 학습 데이터
for i in range(6):  # 6번 호출 (5번 학습 + 1번 완료)
    model.train(train_data)

# 3. 학습 완료 후 예측
print("\n3. 학습 완료 후 예측:")
test_input = np.array([[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]])
prediction = model.predict(test_input)

# 4. 모델 저장
print("\n4. 모델 저장:")
model.save("my_model.pkl")

# 5. 추론 모드로 전환
print("\n5. 추론 모드 전환:")
model.set_inference_mode()

# 6. 추론 모드에서 예측
print("\n6. 추론 모드에서 예측:")
prediction = model.predict(test_input)

print("\n🎯 State 패턴으로 모델 라이프사이클 상태 관리!")
print("   각 상태에서 허용되는 작업이 자동으로 제어됨")

=== 모델 라이프사이클 관리 데모 ===
1. 초기 상태에서 예측 시도:
에러: 모델이 학습되지 않았습니다. 먼저 train()을 호출하세요.

2. 학습 시작:
모델 학습 시작...
모델 상태 변경: TrainingState
✓ 학습 상태로 전환
학습 진행 중... 에폭 1/5
학습 진행 중... 에폭 2/5
학습 진행 중... 에폭 3/5
학습 진행 중... 에폭 4/5
학습 진행 중... 에폭 5/5
모델 상태 변경: TrainedState
✓ 학습 완료! 추론 준비 상태로 전환

3. 학습 완료 후 예측:
예측 결과: [-0.93829674]

4. 모델 저장:
학습된 모델 저장: my_model.pkl

5. 추론 모드 전환:
모델 상태 변경: InferenceState
✓ 추론 모드로 전환

6. 추론 모드에서 예측:
[추론 모드] 예측 결과: [-0.93829674]

🎯 State 패턴으로 모델 라이프사이클 상태 관리!
   각 상태에서 허용되는 작업이 자동으로 제어됨


### 3-9. Strategy 패턴

Strategy 패턴은 **"동일한 목적(알고리즘)을 수행하는 여러 방법을 캡슐화하고 런타임에 선택해 사용하는 패턴"**입니다.

**비유로 이해하기:**
- **내비게이션 경로**: 목적지는 같지만 "최단거리", "고속도로 우선", "톨게이트 피함" 전략 선택
- 같은 출발지/목적지라도 전략에 따라 다른 경로

**코드로 이해하기:**
- `Strategy`: 알고리즘 인터페이스
- `ConcreteStrategy`: 구체적인 알고리즘들
- `Context`: 전략을 사용하는 객체

**머신러닝 엔지니어 관점:**
다양한 알고리즘을 쉽게 교체하고 비교할 때 유용합니다. 예를 들어, 최적화 알고리즘(SGD, Adam, RMSprop)을 전략으로 선택하여 같은 모델에 적용.

**장점:**
- 알고리즘 교체가 쉽고 코드 중복 감소
- 새로운 전략 쉽게 추가
- 전략을 독립적으로 테스트 가능

**단점:**
- 클라이언트가 전략을 알아야 함
- 전략 수가 많아지면 관리 복잡

In [21]:
# Strategy 패턴 더 자세한 예제: 최적화 알고리즘 선택
# 머신러닝에서 Strategy 패턴 적용 예시

from abc import ABC, abstractmethod
import numpy as np

# Strategy 인터페이스
class OptimizerStrategy(ABC):
    @abstractmethod
    def update_weights(self, weights, gradients, learning_rate):
        pass
    
    @abstractmethod
    def get_name(self):
        pass

# Concrete Strategies: 다양한 최적화 알고리즘
class SGDOptimizer(OptimizerStrategy):
    """확률적 경사 하강법 (Stochastic Gradient Descent)"""
    def update_weights(self, weights, gradients, learning_rate):
        # W = W - learning_rate * gradient
        return weights - learning_rate * gradients
    
    def get_name(self):
        return "SGD"

class AdamOptimizer(OptimizerStrategy):
    """Adam 최적화 알고리즘"""
    def __init__(self, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = 0  # 1차 모멘트
        self.v = 0  # 2차 모멘트
        self.t = 0  # 타임 스텝
    
    def update_weights(self, weights, gradients, learning_rate):
        self.t += 1
        
        # 1차 모멘트 업데이트
        self.m = self.beta1 * self.m + (1 - self.beta1) * gradients
        
        # 2차 모멘트 업데이트
        self.v = self.beta2 * self.v + (1 - self.beta2) * (gradients ** 2)
        
        # 바이어스 보정
        m_hat = self.m / (1 - self.beta1 ** self.t)
        v_hat = self.v / (1 - self.beta2 ** self.t)
        
        # 가중치 업데이트
        return weights - learning_rate * m_hat / (np.sqrt(v_hat) + self.epsilon)
    
    def get_name(self):
        return "Adam"

class RMSpropOptimizer(OptimizerStrategy):
    """RMSprop 최적화 알고리즘"""
    def __init__(self, rho=0.9, epsilon=1e-8):
        self.rho = rho
        self.epsilon = epsilon
        self.Eg2 = 0  # 기울기 제곱의 지수 가중 이동 평균
    
    def update_weights(self, weights, gradients, learning_rate):
        # 기울기 제곱의 지수 가중 이동 평균 업데이트
        self.Eg2 = self.rho * self.Eg2 + (1 - self.rho) * (gradients ** 2)
        
        # 가중치 업데이트
        return weights - learning_rate * gradients / (np.sqrt(self.Eg2) + self.epsilon)
    
    def get_name(self):
        return "RMSprop"

# Context: 모델 학습 컨텍스트
class ModelTrainer:
    def __init__(self, optimizer_strategy: OptimizerStrategy, learning_rate=0.01):
        self.optimizer = optimizer_strategy
        self.learning_rate = learning_rate
        self.weights = np.array([0.5, -0.2, 0.8])  # 초기 가중치
        self.loss_history = []
    
    def set_optimizer(self, optimizer_strategy: OptimizerStrategy):
        """런타임에 최적화 전략 변경"""
        self.optimizer = optimizer_strategy
        print(f"최적화 알고리즘 변경: {optimizer_strategy.get_name()}")
    
    def train_step(self, X, y):
        """학습 스텝"""
        # 간단한 선형 회귀 예측
        y_pred = np.dot(X, self.weights)
        
        # MSE 손실 계산
        loss = np.mean((y_pred - y) ** 2)
        self.loss_history.append(loss)
        
        # 기울기 계산 (dL/dW)
        gradients = 2 * np.dot(X.T, (y_pred - y)) / len(X)
        
        # 최적화 알고리즘으로 가중치 업데이트
        self.weights = self.optimizer.update_weights(self.weights, gradients, self.learning_rate)
        
        return loss
    
    def get_weights(self):
        return self.weights.copy()
    
    def get_optimizer_name(self):
        return self.optimizer.get_name()

# 테스트: 다양한 최적화 전략 비교
print("=== 최적화 알고리즘 전략 비교 데모 ===")

# 샘플 데이터 생성
np.random.seed(42)
X = np.random.randn(100, 3)
true_weights = np.array([2.0, -1.5, 3.0])
y = np.dot(X, true_weights) + np.random.randn(100) * 0.1

print(f"실제 가중치: {true_weights}")

# 다양한 최적화 알고리즘으로 학습
optimizers = [
    SGDOptimizer(),
    AdamOptimizer(),
    RMSpropOptimizer()
]

results = {}

for optimizer in optimizers:
    print(f"\n--- {optimizer.get_name()} 최적화 ---")
    
    # 트레이너 생성
    trainer = ModelTrainer(optimizer, learning_rate=0.01)
    
    # 학습
    for epoch in range(50):
        loss = trainer.train_step(X, y)
        if epoch % 10 == 0:
            print(f"에폭 {epoch}: 손실 = {loss:.4f}")
    
    final_weights = trainer.get_weights()
    final_loss = trainer.loss_history[-1]
    
    results[optimizer.get_name()] = {
        'final_weights': final_weights,
        'final_loss': final_loss,
        'weight_error': np.abs(final_weights - true_weights)
    }
    
    print(f"최종 가중치: {final_weights}")
    print(f"가중치 오차: {results[optimizer.get_name()]['weight_error']}")

# 결과 비교
print("\n=== 최적화 알고리즘 비교 ===")
for name, result in results.items():
    print(f"{name}: 최종 손실 = {result['final_loss']:.4f}, "
          f"평균 오차 = {np.mean(result['weight_error']):.4f}")

# 런타임 전략 변경 데모
print("\n=== 런타임 전략 변경 데모 ===")
trainer = ModelTrainer(SGDOptimizer(), learning_rate=0.01)
print(f"초기 최적화: {trainer.get_optimizer_name()}")

# Adam으로 변경
trainer.set_optimizer(AdamOptimizer())
print(f"변경 후 최적화: {trainer.get_optimizer_name()}")

# 추가 학습
for epoch in range(10):
    loss = trainer.train_step(X, y)

print(f"Adam으로 추가 학습 후 손실: {loss:.4f}")

print("\n🎯 Strategy 패턴으로 최적화 알고리즘 유연하게 교체!")
print("   같은 모델에 다른 최적화 전략 적용 및 비교 가능")

=== 최적화 알고리즘 전략 비교 데모 ===
실제 가중치: [ 2.  -1.5  3. ]

--- SGD 최적화 ---
에폭 0: 손실 = 9.5017
에폭 10: 손실 = 5.9564
에폭 20: 손실 = 3.7813
에폭 30: 손실 = 2.4349
에폭 40: 손실 = 1.5926
최종 가중치: [ 1.18100114 -1.16876227  2.35183808]
가중치 오차: [0.81899886 0.33123773 0.64816192]

--- Adam 최적화 ---
에폭 0: 손실 = 9.5017
에폭 10: 손실 = 8.4762
에폭 20: 손실 = 7.5225
에폭 30: 손실 = 6.6465
에폭 40: 손실 = 5.8498
최종 가중치: [ 0.97453868 -0.67568953  1.28463763]
가중치 오차: [1.02546132 0.82431047 1.71536237]

--- RMSprop 최적화 ---
에폭 0: 손실 = 9.5017
에폭 10: 손실 = 7.7893
에폭 20: 손실 = 6.7987
에폭 30: 손실 = 5.9567
에폭 40: 손실 = 5.1985
최종 가중치: [ 1.0572859  -0.7587927   1.37112338]
가중치 오차: [0.9427141  0.7412073  1.62887662]

=== 최적화 알고리즘 비교 ===
SGD: 최종 손실 = 1.1025, 평균 오차 = 0.5995
Adam: 최종 손실 = 5.1993, 평균 오차 = 1.1884
RMSprop: 최종 손실 = 4.5725, 평균 오차 = 1.1043

=== 런타임 전략 변경 데모 ===
초기 최적화: SGD
최적화 알고리즘 변경: Adam
변경 후 최적화: Adam
Adam으로 추가 학습 후 손실: 8.5757

🎯 Strategy 패턴으로 최적화 알고리즘 유연하게 교체!
   같은 모델에 다른 최적화 전략 적용 및 비교 가능


### 3-10. Template Method 패턴

Template Method 패턴은 **"알고리즘의 골격을 상위 클래스에 정의하고, 일부 단계를 하위 클래스에서 구현하는 패턴"**입니다.

**비유로 이해하기:**
- **요리 레시피**: "재료 준비 → 조리 → 서빙" 골격은 같지만, 각 요리별로 구체적인 단계가 다름
- 볶음밥, 파스타 모두 같은 골격이지만 재료 준비와 조리 방법이 다름

**코드로 이해하기:**
- `AbstractClass`: 알고리즘 골격 정의 (template_method)
- `ConcreteClass`: 구체적인 단계 구현

**머신러닝 엔지니어 관점:**
ML 파이프라인의 표준화된 골격을 정의할 때 유용합니다. 예를 들어, "데이터 로드 → 전처리 → 모델 학습 → 평가" 골격은 같지만, 각 프로젝트에서 구체적인 구현이 다름.

**장점:**
- 중복 코드 감소, 알고리즘 골격 통제
- 코드 재사용성 향상
- 변경에 대한 유연성

**단점:**
- 상속 구조 고정으로 유연성 제한
- 골격 변경 시 모든 서브클래스 영향

In [ ]:
# Template Method 패턴 더 자세한 예제: ML 실험 파이프라인
# 머신러닝에서 Template Method 패턴 적용 예시

from abc import ABC, abstractmethod
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

# Abstract Class: ML 실험 골격
class MLExperiment(ABC):
    def run_experiment(self):
        """템플릿 메서드: ML 실험의 표준 골격"""
        print("=== ML 실험 시작 ===")
        
        # 1. 데이터 로드 (공통)
        data = self.load_data()
        print(f"✓ 데이터 로드 완료: {data.shape}")
        
        # 2. 데이터 전처리 (서브클래스 구현)
        processed_data = self.preprocess_data(data)
        print("✓ 데이터 전처리 완료")
        
        # 3. 모델 학습 (서브클래스 구현)
        model = self.train_model(processed_data)
        print("✓ 모델 학습 완료")
        
        # 4. 모델 평가 (공통 + 서브클래스 확장)
        metrics = self.evaluate_model(model, processed_data)
        print("✓ 모델 평가 완료")
        
        # 5. 결과 보고 (서브클래스 구현)
        self.report_results(metrics)
        print("✓ 실험 완료 ===\n")
        
        return model, metrics
    
    def load_data(self):
        """공통 데이터 로드 로직"""
        # 샘플 데이터 생성 (실제로는 파일/DB에서 로드)
        np.random.seed(42)
        n_samples = 1000
        data = pd.DataFrame({
            'feature1': np.random.randn(n_samples),
            'feature2': np.random.randn(n_samples),
            'feature3': np.random.randn(n_samples),
            'target': np.random.randint(0, 3, n_samples)  # 다중 클래스
        })
        return data
    
    def evaluate_model(self, model, data):
        """공통 평가 로직"""
        # 간단한 평가 (훈련 데이터로 - 실제로는 검증/테스트 세트 사용)
        X = data['X']
        y_true = data['y']
        
        # 모델 예측 (서브클래스에서 구현한 predict 메서드 사용)
        y_pred = self.predict_model(model, X)
        
        # 메트릭 계산
        accuracy = accuracy_score(y_true, y_pred)
        report = classification_report(y_true, y_pred, output_dict=True)
        
        return {
            'accuracy': accuracy,
            'classification_report': report
        }
    
    @abstractmethod
    def preprocess_data(self, data):
        """데이터 전처리 (서브클래스 구현)"""
        pass
    
    @abstractmethod
    def train_model(self, data):
        """모델 학습 (서브클래스 구현)"""
        pass
    
    @abstractmethod
    def predict_model(self, model, X):
        """모델 예측 (서브클래스 구현)"""
        pass
    
    @abstractmethod
    def report_results(self, metrics):
        """결과 보고 (서브클래스 구현)"""
        pass

# Concrete Class 1: 의사결정나무 실험
class DecisionTreeExperiment(MLExperiment):
    def preprocess_data(self, data):
        """의사결정나무용 전처리: 스케일링 불필요"""
        X = data[['feature1', 'feature2', 'feature3']].values
        y = data['target'].values
        return {'X': X, 'y': y}
    
    def train_model(self, data):
        """의사결정나무 학습"""
        from sklearn.tree import DecisionTreeClassifier
        model = DecisionTreeClassifier(max_depth=5, random_state=42)
        model.fit(data['X'], data['y'])
        return model
    
    def predict_model(self, model, X):
        """의사결정나무 예측"""
        return model.predict(X)
    
    def report_results(self, metrics):
        """의사결정나무 결과 보고"""
        print("의사결정나무 실험 결과:")
        print(f"정확도: {metrics['accuracy']:.4f}")
        print("상세 리포트:")
        for label, scores in metrics['classification_report'].items():
            if isinstance(scores, dict):
                print(f"  클래스 {label}: 정밀도={scores['precision']:.3f}, 재현율={scores['recall']:.3f}")

# Concrete Class 2: 랜덤포레스트 실험
class RandomForestExperiment(MLExperiment):
    def preprocess_data(self, data):
        """랜덤포레스트용 전처리: 스케일링 불필요"""
        X = data[['feature1', 'feature2', 'feature3']].values
        y = data['target'].values
        return {'X': X, 'y': y}
    
    def train_model(self, data):
        """랜덤포레스트 학습"""
        from sklearn.ensemble import RandomForestClassifier
        model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
        model.fit(data['X'], data['y'])
        return model
    
    def predict_model(self, model, X):
        """랜덤포레스트 예측"""
        return model.predict(X)
    
    def report_results(self, metrics):
        """랜덤포레스트 결과 보고"""
        print("랜덤포레스트 실험 결과:")
        print(f"정확도: {metrics['accuracy']:.4f}")
        print("특성 중요도:")
        # 실제로는 model.feature_importances_ 출력
        print("  feature1: 0.35, feature2: 0.40, feature3: 0.25")

# Concrete Class 3: SVM 실험 (스케일링 필요)
class SVMExperiment(MLExperiment):
    def preprocess_data(self, data):
        """SVM용 전처리: 스케일링 필요"""
        from sklearn.preprocessing import StandardScaler
        
        X = data[['feature1', 'feature2', 'feature3']].values
        y = data['target'].values
        
        # 스케일링
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        return {'X': X_scaled, 'y': y, 'scaler': scaler}
    
    def train_model(self, data):
        """SVM 학습"""
        from sklearn.svm import SVC
        model = SVC(kernel='rbf', C=1.0, random_state=42)
        model.fit(data['X'], data['y'])
        return model
    
    def predict_model(self, model, X):
        """SVM 예측 (스케일링 적용)"""
        # 실제로는 저장된 scaler 사용해야 함
        return model.predict(X)
    
    def report_results(self, metrics):
        """SVM 결과 보고"""
        print("SVM 실험 결과:")
        print(f"정확도: {metrics['accuracy']:.4f}")
        print("커널: RBF, C: 1.0")

# 테스트: 다양한 ML 실험 실행
print("=== ML 실험 템플릿 데모 ===")

experiments = [
    DecisionTreeExperiment(),
    RandomForestExperiment(),
    SVMExperiment()
]

results = {}
for experiment in experiments:
    model_name = experiment.__class__.__name__.replace('Experiment', '')
    print(f"\n--- {model_name} 실험 ---")
    
    model, metrics = experiment.run_experiment()
    results[model_name] = metrics['accuracy']

# 결과 비교
print("=== 실험 결과 비교 ===")
for model_name, accuracy in results.items():
    print(f"{model_name}: 정확도 = {accuracy:.4f}")

print("\n🎯 Template Method 패턴으로 표준화된 ML 실험 파이프라인!")
print("   골격은 같지만 각 알고리즘별 구체적 구현이 다름")

=== ML 실험 템플릿 데모 ===

--- DecisionTree 실험 ---
=== ML 실험 시작 ===
✓ 데이터 로드 완료: (1000, 4)
✓ 데이터 전처리 완료
✓ 모델 학습 완료


TypeError: unhashable type: 'list'

### 3-11. Visitor 패턴

Visitor 패턴은 **"객체 구조에 대해 새로운 연산을 추가할 때, 연산을 방문자로 분리해 요소들을 방문하며 연산 수행하는 패턴"**입니다.

**비유로 이해하기:**
- **병원 진료**: 환자가 여러 진료과를 방문
- 각 과마다 다른 검사/치료를 하지만, 방문 순회는 같음
- 새로운 진료과(연산) 쉽게 추가 가능

**코드로 이해하기:**
- `Element`: 방문 가능한 요소 인터페이스 (accept 메서드)
- `Visitor`: 연산 인터페이스 (visit_* 메서드)
- `ConcreteVisitor`: 구체적인 연산들

**머신러닝 엔지니어 관점:**
모델 구조를 순회하면서 다양한 연산을 수행할 때 유용합니다. 예를 들어, 신경망의 각 레이어를 방문하여 가중치 초기화, 그래디언트 계산, 모델 시각화 등을 수행.

**장점:**
- 새로운 연산 쉽게 추가 (요소 클래스 수정 불필요)
- 관련 연산을 한 곳에 모음
- 단일 책임 원칙 준수

**단점:**
- 요소 구조 변경 시 모든 visitor 수정 필요
- 캡슐화 약화 (내부 구조 노출)

In [ ]:
# Visitor 패턴 더 자세한 예제: 신경망 모델 순회 및 연산
# 머신러닝에서 Visitor 패턴 적용 예시

from abc import ABC, abstractmethod
import numpy as np

# Element 인터페이스: 방문 가능한 레이어
class Layer(ABC):
    @abstractmethod
    def accept(self, visitor):
        pass
    
    @abstractmethod
    def get_name(self):
        pass

# Concrete Elements: 다양한 레이어 타입
class DenseLayer(Layer):
    def __init__(self, units, input_shape=None):
        self.units = units
        self.input_shape = input_shape
        self.weights = None
        self.bias = None
    
    def accept(self, visitor):
        visitor.visit_dense_layer(self)
    
    def get_name(self):
        return f"Dense({self.units})"

class Conv2DLayer(Layer):
    def __init__(self, filters, kernel_size, input_shape=None):
        self.filters = filters
        self.kernel_size = kernel_size
        self.input_shape = input_shape
        self.kernels = None
        self.bias = None
    
    def accept(self, visitor):
        visitor.visit_conv2d_layer(self)
    
    def get_name(self):
        return f"Conv2D({self.filters}, {self.kernel_size}x{self.kernel_size})"

class ActivationLayer(Layer):
    def __init__(self, activation_type):
        self.activation_type = activation_type
    
    def accept(self, visitor):
        visitor.visit_activation_layer(self)
    
    def get_name(self):
        return f"Activation({self.activation_type})"

class DropoutLayer(Layer):
    def __init__(self, rate):
        self.rate = rate
    
    def accept(self, visitor):
        visitor.visit_dropout_layer(self)
    
    def get_name(self):
        return f"Dropout({self.rate})"

# Visitor 인터페이스: 레이어 연산들
class LayerVisitor(ABC):
    @abstractmethod
    def visit_dense_layer(self, layer: DenseLayer):
        pass
    
    @abstractmethod
    def visit_conv2d_layer(self, layer: Conv2DLayer):
        pass
    
    @abstractmethod
    def visit_activation_layer(self, layer: ActivationLayer):
        pass
    
    @abstractmethod
    def visit_dropout_layer(self, layer: DropoutLayer):
        pass

# Concrete Visitors: 다양한 연산들
class WeightInitializerVisitor(LayerVisitor):
    """가중치 초기화 방문자"""
    def __init__(self, init_method="xavier"):
        self.init_method = init_method
    
    def visit_dense_layer(self, layer: DenseLayer):
        if layer.input_shape:
            if self.init_method == "xavier":
                # Xavier 초기화
                limit = np.sqrt(6 / (layer.input_shape + layer.units))
                layer.weights = np.random.uniform(-limit, limit, (layer.input_shape, layer.units))
            else:
                # 랜덤 초기화
                layer.weights = np.random.randn(layer.input_shape, layer.units) * 0.1
            
            layer.bias = np.zeros(layer.units)
            print(f"✓ {layer.get_name()} 가중치 초기화: {layer.weights.shape}")
    
    def visit_conv2d_layer(self, layer: Conv2DLayer):
        if layer.input_shape:
            # Conv2D 가중치 초기화 (커널)
            if self.init_method == "xavier":
                limit = np.sqrt(6 / (np.prod(layer.input_shape) + layer.filters))
                layer.kernels = np.random.uniform(-limit, limit, 
                    (layer.filters, layer.input_shape[-1], layer.kernel_size, layer.kernel_size))
            else:
                layer.kernels = np.random.randn(layer.filters, layer.input_shape[-1], 
                    layer.kernel_size, layer.kernel_size) * 0.1
            
            layer.bias = np.zeros(layer.filters)
            print(f"✓ {layer.get_name()} 커널 초기화: {layer.kernels.shape}")
    
    def visit_activation_layer(self, layer: ActivationLayer):
        print(f"✓ {layer.get_name()} 활성화 함수 설정 완료")
    
    def visit_dropout_layer(self, layer: DropoutLayer):
        print(f"✓ {layer.get_name()} 드롭아웃 설정 완료")

class ParameterCounterVisitor(LayerVisitor):
    """파라미터 수 계산 방문자"""
    def __init__(self):
        self.total_params = 0
    
    def visit_dense_layer(self, layer: DenseLayer):
        if layer.weights is not None:
            params = layer.weights.size + layer.bias.size
            self.total_params += params
            print(f"✓ {layer.get_name()} 파라미터: {params:,}")
    
    def visit_conv2d_layer(self, layer: Conv2DLayer):
        if layer.kernels is not None:
            params = layer.kernels.size + layer.bias.size
            self.total_params += params
            print(f"✓ {layer.get_name()} 파라미터: {params:,}")
    
    def visit_activation_layer(self, layer: ActivationLayer):
        print(f"✓ {layer.get_name()} 파라미터: 0")
    
    def visit_dropout_layer(self, layer: DropoutLayer):
        print(f"✓ {layer.get_name()} 파라미터: 0")

class GradientComputerVisitor(LayerVisitor):
    """그래디언트 계산 방문자"""
    def __init__(self, learning_rate=0.01):
        self.learning_rate = learning_rate
    
    def visit_dense_layer(self, layer: DenseLayer):
        if layer.weights is not None:
            # 가상의 그래디언트 (실제로는 역전파로 계산)
            grad_w = np.random.randn(*layer.weights.shape) * 0.01
            grad_b = np.random.randn(*layer.bias.shape) * 0.01
            
            # 가중치 업데이트
            layer.weights -= self.learning_rate * grad_w
            layer.bias -= self.learning_rate * grad_b
            
            print(f"✓ {layer.get_name()} 그래디언트 업데이트 완료")
    
    def visit_conv2d_layer(self, layer: Conv2DLayer):
        if layer.kernels is not None:
            # 가상의 그래디언트
            grad_k = np.random.randn(*layer.kernels.shape) * 0.01
            grad_b = np.random.randn(*layer.bias.shape) * 0.01
            
            # 커널 업데이트
            layer.kernels -= self.learning_rate * grad_k
            layer.bias -= self.learning_rate * grad_b
            
            print(f"✓ {layer.get_name()} 그래디언트 업데이트 완료")
    
    def visit_activation_layer(self, layer: ActivationLayer):
        print(f"✓ {layer.get_name()} 그래디언트 계산 (활성화 함수)")
    
    def visit_dropout_layer(self, layer: DropoutLayer):
        print(f"✓ {layer.get_name()} 드롭아웃 적용 (학습 시)")

# Neural Network: 레이어들을 관리하고 visitor를 받아들임
class NeuralNetwork:
    def __init__(self):
        self.layers = []
    
    def add_layer(self, layer: Layer):
        self.layers.append(layer)
    
    def accept(self, visitor: LayerVisitor):
        """모든 레이어를 방문자로 순회"""
        for layer in self.layers:
            layer.accept(visitor)

# 테스트: 신경망에 다양한 visitor 적용
print("=== 신경망 Visitor 패턴 데모 ===")

# 신경망 모델 구성
model = NeuralNetwork()
model.add_layer(DenseLayer(128, input_shape=784))  # 입력: 28x28 이미지 평탄화
model.add_layer(ActivationLayer("relu"))
model.add_layer(DropoutLayer(0.2))
model.add_layer(DenseLayer(64))
model.add_layer(ActivationLayer("relu"))
model.add_layer(DropoutLayer(0.2))
model.add_layer(DenseLayer(10))  # 출력: 10개 클래스
model.add_layer(ActivationLayer("softmax"))

print("신경망 구조:")
for i, layer in enumerate(model.layers):
    print(f"  {i+1}. {layer.get_name()}")

# 1. 가중치 초기화
print("\n--- 가중치 초기화 ---")
initializer = WeightInitializerVisitor(init_method="xavier")
model.accept(initializer)

# 2. 파라미터 수 계산
print("\n--- 파라미터 수 계산 ---")
counter = ParameterCounterVisitor()
model.accept(counter)
print(f"총 파라미터 수: {counter.total_params:,}")

# 3. 그래디언트 계산 (학습 시뮬레이션)
print("\n--- 그래디언트 계산 ---")
gradient_computer = GradientComputerVisitor(learning_rate=0.001)
model.accept(gradient_computer)

print("\n🎯 Visitor 패턴으로 신경망 레이어에 다양한 연산 적용!")
print("   새로운 연산(예: 양자화, 프루닝)을 visitor로 쉽게 추가 가능")

---
## 정리

### 생성 패턴 (Creational)
- **Singleton**: 객체 하나만 필요할 때
- **Factory Method**: 어떤 종류의 객체를 선택해서 만들어야 할 때
- **Abstract Factory**: 서로 관련된 여러 객체(세트)를 만들어야 할 때
- **Builder**: 복잡한 객체를 단계적으로 만들고 싶을 때
- **Prototype**: 이미 만들어진 객체를 복사해서 빠르게 만들고 싶을 때

### 구조 패턴 (Structural)
- **Adapter**: 인터페이스가 다른 두 객체를 연결할 때
- **Bridge**: 추상과 구현을 분리해서 독립적으로 바꿀 때
- **Composite**: 개별 객체와 여러 객체 묶음을 같은 방식으로 다룰 때
- **Decorator**: 객체에 기능을 런타임에 유연하게 추가할 때
- **Facade**: 복잡한 서브시스템을 단순한 인터페이스로 감쌀 때
- **Flyweight**: 거의 동일한 객체를 아주 많이 만들어 메모리가 문제될 때
- **Proxy**: 객체 접근을 제어하거나 지연 생성이 필요할 때

### 행동 패턴 (Behavioral)
- **Chain of Responsibility**: 요청을 여러 객체가 차례로 처리할 때
- **Command**: 작업을 객체로 캡슐화하고 실행/큐잉/취소가 필요할 때
- **Interpreter**: 도메인 전용 언어를 해석해야 할 때
- **Iterator**: 컬렉션을 순회하는 방법을 캡슐화할 때
- **Mediator**: 객체들 간의 복잡한 통신을 중앙에서 관리할 때
- **Memento**: 객체의 상태를 캡슐화해서 나중에 복원할 때
- **Observer**: 한 객체의 상태 변화를 여러 관찰자에게 알릴 때
- **State**: 객체의 내부 상태에 따라 행동을 바꿀 때
- **Strategy**: 알고리즘을 런타임에 바꿔야 할 때
- **Template Method**: 알고리즘 골격은 유지하되 일부 단계만 바꿀 때
- **Visitor**: 객체 구조를 순회하면서 새로운 연산을 추가할 때